In [ ]:
!git pull

# Diagnostics before the rollout

What the sanity checks (`sanity.ipynb`, steps 1-9) leave open, measured on Colab at batch 128, CLS projector, direct
predictor, lr 1e-4, 1000 steps, dropout off, as in steps 7-8. Every run is saved to Drive (metrics and weights) and
reloaded instead of retrained when its cell is rerun, so a disconnected session loses at most the run in progress.

**The base changed after E3: β 1e-3 → 1e-4.** At 1e-3 the KL starved z (tf/pers 0.75 against 0.40, z carries motion,
not the next frame). The runs of the β 1e-3 base (E1 with five seeds, E2-E5) stay in the results folder; the new base
writes in its subfolder `beta-1e-4`, where the four runs of E3 are carried over (the same configs) and the rest runs again.

| | question | runs | cost (A100) |
|---|---|---|---|
| E1 | how much do two runs differ by chance alone? | base, seeds 0-2 (seed 0 = β 1e-4 of E3) | 2 x ~10 min |
| E2 | the invented motion on static clips: a constant drift of the predictor? | none (net of E1) | seconds |
| E3 | the under-prediction (x0.5) on moving clips: is the KL the bottleneck? does z carry motion or content? | β 0 / 1e-3 / 1e-2; probes of the SSv2 class from z | carried over |
| E4 | the SIGReg weight, taken from LeWM, never set on our data | λ 0.03 / 0.27 | 2 x ~10 min |
| E5 | does the encoder see the hand? trained ViT-tiny vs frozen DINOv2 | DINOv2 | ~10 min |
| E6 | are we limited by data, and by which kind: more of the same or more actions? | 5h x 1/4, 1/2, 1; 10h; 10h 20 classes | 5 x ~25 min |
| E7 | is the single CLS vector the bottleneck? 16 patches (hidden 2048 / 192), 4 tokens, 16 patches on frozen DINOv2 | 4 | 4 x ~12 min |
| E8 | the rollout loss (K = 2): does it fix the problems of E1-E5, which new ones does it bring, how do the rollouts behave? | rollout K2, seeds 0-2 | 3 x ~15 min |

Every probe adds three things to the metrics of training: the metrics per tercile of motion (the static transitions,
half of the data, hide the moving ones in the means), a next-frame retrieval (is the prediction closer to s_t+1 than to
the other frames of its clip, blind to the scale of the embeddings) and, at the end, 95 % intervals by bootstrap over
videos (the val clips are overlapping windows of fewer videos). Every comparison is read against the seed noise of E1.

In [ ]:
# colab: the code of the data-scale branch, the three SSv2 datasets from the Hub (private: HF token), Drive for the
# results. Skip on the Mac. The package is not installed (it asks for Python >= 3.13): src goes on the import path.
import os
import sys
from getpass import getpass

%cd /content
![ -d human-to-robot ] || git clone https://github.com/batgr/human-to-robot
%cd /content/human-to-robot
!git fetch -q && git checkout -q data-scale && git pull -q
!pip install -q einops umap-learn
sys.path.insert(0, "src")

from google.colab import drive
from huggingface_hub import snapshot_download

drive.mount("/content/drive")
os.environ["HF_TOKEN"] = getpass("HF token: ")
for name in ["5h", "10h", "10h-20cls"]:
    snapshot_download(f"batgre/h2r-ssv2-{name}", repo_type="dataset", local_dir=f"/content/h2r-ssv2-{name}")

In [ ]:
# Setup: config, the val clips every run is measured on (5h val: it is inside the val of the bigger datasets, the split
# is a hash of the video id), the training streams. LeWM's lr and SIGReg weight are set for cfg["batch_size"] (128),
# both follow the batch: lr by the square root of the ratio, the SIGReg weight by its inverse.
import copy
import os
import shutil
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
import torch
from torch.utils.data import DataLoader, Subset

from h2r.cli import load, override
from h2r.data.dataset import ClipDataset, shards
from h2r.models.jepa import build_model
from h2r.models.module import SIGReg
from h2r.training.utils import forever, make_autocast, pick_device, seed_everything

COLAB = os.path.exists("/content")
ROOTS = {name: f"/content/h2r-ssv2-{name}" if COLAB else f"/Volumes/PortableSSD/h2r-ssv2/processed-{name}"
         for name in ["5h", "10h", "10h-20cls"]}
RESULTS = Path("/content/drive/MyDrive/h2r-diagnostics" if COLAB else "outputs/diagnostics")  # runs of the β 1e-3 base
OUT = RESULTS / "beta-1e-4"  # one .pt per run of the β 1e-4 base
OUT.mkdir(parents=True, exist_ok=True)
for new, old in {"base": "β 1e-4", "β 0": "β 0", "β 1e-3": "base", "β 1e-2": "β 1e-2"}.items():  # E3: the same configs
    if not (OUT / f"{new}.pt").exists() and (RESULTS / f"{old}.pt").exists():
        shutil.copy(RESULTS / f"{old}.pt", OUT / f"{new}.pt")

cfg = load("configs/training/world_model.yaml")
cfg["loss"]["rollout_steps"] = 1  # E1-E7: teacher forcing only, as they ran (the repo default is K = 2 since the rollout loss); E8 sets it per run
device = pick_device()
autocast = make_autocast(cfg["precision"], device)
torch.set_float32_matmul_precision("high")
T = cfg["data"]["clip_len"]
BATCH = 128 if device.type == "cuda" else 32  # LeWM's batch on the GPU, what fits on the Mac
STEPS, PROBE, VAL_BATCHES = 1000, 50, 8
sigreg = SIGReg().to(device)

ds_val = ClipDataset(ROOTS["5h"], "val", T)
order = torch.randperm(len(ds_val), generator=torch.Generator().manual_seed(0)).tolist()[: VAL_BATCHES * BATCH]
val_batches = list(DataLoader(ds_val, batch_size=BATCH, sampler=order))
val_video = np.repeat(np.searchsorted(ds_val.ends, order, side="right"), T - 1)  # video of each val transition


def videos(ds):
    "First clip index of each video of a ClipDataset, and one past its last"
    return np.r_[0, ds.ends[:-1]], ds.ends


def hours(ds, idx=None):
    lengths = ds.frames.value_lengths().to_numpy()
    return (lengths if idx is None else lengths[idx]).sum() / cfg["data"]["fps"] / 3600


def train_set(data, fraction=1.0):
    "The train split of a dataset, or the clips of a fraction of its videos (fixed draw: 1/4 is inside 1/2)"
    ds = ClipDataset(ROOTS[data], "train", T)
    if fraction == 1.0:
        return ds, hours(ds)
    n = len(ds.ends)
    kept = np.sort(np.random.default_rng(0).permutation(n)[: round(fraction * n)])
    start, end = videos(ds)
    return Subset(ds, np.concatenate([np.arange(start[v], end[v]) for v in kept])), hours(ds, kept)


def train_batches(ds, seed):
    loader = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=True, generator=torch.Generator().manual_seed(seed),
                        num_workers=cfg["num_workers"], persistent_workers=cfg["num_workers"] > 0)
    return forever(loader)


for name, root in ROOTS.items():
    if shards(root, "train"):
        print(f"{name}: train {hours(ClipDataset(root, 'train', T)):.2f} h of frames")
print(f"{device} | batch {BATCH} | val: {len(order)} clips of {len(np.unique(val_video))} videos | results in {OUT}")

In [ ]:
# Tools. A run: a fresh model (its seed sets the init and the data order) trained STEPS steps, dropout off, the lr
# schedule of the real training (warmup over 10 % of the steps then cosine decay to 0: cs231n, anneal the lr; the
# values at the end are less noisy than at a constant lr, as in sanity), the config of the base run (cfg, lr 1e-4) with some values overridden as on the command line
# ("loss.beta=0", values at batch 128), on a dataset or a fraction of it. Every probe steps: the validation metrics of
# training (evaluate, eval mode, z = mu) and the diagnostics of sanity step 3c, plus the metrics per tercile of motion and
# the next-frame retrieval. Saved to OUT/<label>.pt with the weights; rerunning a cell reloads it.
# Diagnostics (diag/, z sampled as in training): the error over persistence, overall and per tercile of motion (static /
# middle / moving: 1 - cos(s_t, s_t+1) of the model, cut at its thirds), the signal of z (spread of mu over sigma), the
# AdaLN gates, the update:data ratios, the share of the predictor's blocks, the predicted change Δpred = pred - s_t
# against the true one Δtrue = s_t+1 - s_t (cosine: direction, norm ratio: size, above 1 = invented motion), the
# next-frame retrieval (the frame of the clip closest to the prediction is s_t+1: persistence 0, chance 1/T), the
# error over FPP3's drift method (s_t + the clip's mean change so far: a second naive benchmark, for moving clips) and
# the train-val gap (the same error on GAP_BATCHES fixed batches of the run's own training clips).
import itertools
import math
import time

import matplotlib.pyplot as plt
import torch.nn.functional as F

from h2r.metrics.prediction import PredictionQuality
from h2r.training.utils import peak_memory_gb
from h2r.training.world_model import build_optimizer, evaluate, train_step, trainable_state

GAP_BATCHES, LR = 2, 1e-4
TERCILES = ["static", "middle", "moving"]
KEY = ["diag/tf_z_sampled / persistence", "diag/tf_z_sampled / persistence, static",
       "diag/tf_z_sampled / persistence, moving", "gap/tf_z_sampled / persistence, train clips",
       "diag/|Δpred| / |Δtrue|, static", "diag/|Δpred| / |Δtrue|, moving", "diag/cos(Δpred, Δtrue)",
       "diag/next-frame retrieval, moving", "diag/tf_z_sampled / drift method, moving", "latent_action/kl",
       "latent_action/active_dims", "diag/mu_spread / sigma",
       "latent_action/controllability", "latent_action/cycle", "representation/temporal_change", "representation/rankme",
       "diag/cos(s_t, s_t+1)", "prediction/tf / persistence", "loss/total_val"]
k = BATCH / cfg["batch_size"]


def run_config(over=()):
    run = override(copy.deepcopy(cfg), [f"optim.lr={LR}", *over])
    run["optim"]["lr"] *= k**0.5
    run["loss"]["sigreg"] /= k
    return run


@torch.no_grad()
def blocks_share(m, clip):
    "|x_L - x_0| / |x_0| in the predictor's residual stream (eval): what its blocks add to the input states"
    xs = []
    hooks = [m.predictor.transformer.layers[0].register_forward_pre_hook(lambda mod, args: xs.append(args[0].float())),
             m.predictor.transformer.norm.register_forward_hook(lambda mod, i, o: xs.append(i[0].float()))]
    m.eval()
    with autocast:
        emb = m.encode(clip)
        m.predict(emb[:, :-1], m.idm(emb)[0])
    m.train()
    for h in hooks:
        h.remove()
    x0, xl = xs
    return ((xl - x0).norm() / x0.norm()).item()


@torch.no_grad()
def diagnostics(m, clips):
    "On a list of clip batches, eval mode except the IDM (z sampled as in training)"
    head, gates = [], []
    hooks = [m.idm.head.register_forward_hook(lambda mod, i, o: head.append(o.float()))]
    hooks += [b.adaLN_modulation.register_forward_hook(lambda mod, i, o: gates.append(o.float().chunk(6, -1)[2::3]))
              for b in m.predictor.transformer.layers]
    m.eval()
    m.idm.train()
    torch.manual_seed(0)
    q, raw = PredictionQuality(), {k: [] for k in ["cos", "error", "persistence", "drift", "change_cos", "change_ratio", "retrieval"]}
    for clip in clips:
        clip = clip.to(device)
        with autocast:
            emb = m.encode(clip)
            z = m.idm(emb)[0]
            pred = m.predict(emb[:, :-1], z)
            q.update(emb, pred, clip)
        e, pf = emb.float().flatten(2), pred.float().flatten(2)  # one vector per frame, its tokens concatenated
        t = torch.arange(T - 1, device=e.device).clamp_min(1)[None, :, None]
        drift = e[:, :-1] + (e[:, :-1] - e[:, :1]) / t  # FPP3's drift method: the clip's mean change so far, persistence at t = 0
        d_true, d_pred = e[:, 1:] - e[:, :-1], pf - e[:, :-1]
        new = {"cos": F.cosine_similarity(e[:, :-1], e[:, 1:], dim=-1),
               "error": (pf - e[:, 1:]).pow(2).mean(-1),
               "persistence": d_true.pow(2).mean(-1),
               "drift": (drift - e[:, 1:]).pow(2).mean(-1),
               "change_cos": F.cosine_similarity(d_pred, d_true, dim=-1),
               "change_ratio": d_pred.norm(dim=-1) / d_true.norm(dim=-1).clamp_min(1e-8),
               "retrieval": (torch.cdist(pf, e).argmin(-1) == torch.arange(1, T, device=e.device)).float()}
        for key, v in new.items():
            raw[key].append(v.flatten().cpu())
    m.train()
    for h in hooks:
        h.remove()
    p = q.compute()
    raw = {key: torch.cat(v).numpy() for key, v in raw.items()}
    mu, logvar = torch.cat(head).chunk(2, -1)
    sigma = (0.5 * logvar.clamp(-10, 10)).exp()
    rank = lambda x: x.argsort().argsort()
    third = tercile(raw["cos"])
    out = {
        "diag/tf_z_sampled / persistence": p["tf"] / p["persistence"],
        "diag/std_ratio_z_sampled": p["std_ratio"],
        "diag/mu_spread / sigma": (mu.std(0) / sigma.mean(0)).mean().item(),
        "diag/sigma": sigma.mean().item(),
        "diag/gate_msa": float(np.mean([g[0].abs().mean().item() for g in gates])),
        "diag/gate_mlp": float(np.mean([g[1].abs().mean().item() for g in gates])),
        "diag/blocks_share": blocks_share(m, clips[0].to(device)),
        "diag/cos(s_t, s_t+1)": raw["cos"].mean(),
        "diag/spearman(1 - cos, error)": np.corrcoef(rank(-raw["cos"]), rank(raw["error"]))[0, 1],
        "diag/cos(Δpred, Δtrue)": raw["change_cos"].mean(),
        "diag/|Δpred| / |Δtrue|": np.median(raw["change_ratio"]),
        "diag/next-frame retrieval": raw["retrieval"].mean(),
        "diag/tf_z_sampled / drift method": raw["error"].sum() / raw["drift"].sum(),
    }
    for i, name in enumerate(TERCILES):
        b = third == i
        out[f"diag/tf_z_sampled / persistence, {name}"] = raw["error"][b].sum() / raw["persistence"][b].sum()
        out[f"diag/|Δpred| / |Δtrue|, {name}"] = np.median(raw["change_ratio"][b])
        out[f"diag/next-frame retrieval, {name}"] = raw["retrieval"][b].mean()
        out[f"diag/tf_z_sampled / drift method, {name}"] = raw["error"][b].sum() / raw["drift"][b].sum()
    return out | {f"raw/{key}": v for key, v in raw.items()}


def tercile(cos):
    "0 static, 1 middle, 2 moving: the thirds of 1 - cos(s_t, s_t+1)"
    return np.digitize(-cos, np.quantile(-cos, [1 / 3, 2 / 3]))


def probe_all(m, run, gap):
    r = evaluate(m, sigreg, val_batches, run, device, autocast)
    g = diagnostics(m, gap)["diag/tf_z_sampled / persistence"]
    return r | {"prediction/tf / persistence": r["prediction/tf"] / r["prediction/persistence"],
                "gap/tf_z_sampled / persistence, train clips": g} | diagnostics(m, val_batches)


def update_ratios(m, before):
    "Median over the weight matrices of each module of log10 std(Δw) / std(w) for one step"
    r = {}
    for n, p in m.named_parameters():
        if n in before and before[n].float().std() > 0:
            r.setdefault(n.split(".")[0], []).append(((p.detach() - before[n]).float().std() / before[n].float().std()).log10().item())
    return {f"diag/update_ratio_{key}": float(np.median(v)) for key, v in r.items()}


def no_dropout(m):
    for mod in m.modules():
        if isinstance(mod, torch.nn.Dropout):
            mod.p = 0.0
        elif isinstance(getattr(mod, "dropout", None), float):  # attention dropout
            mod.dropout = 0.0
    return m


def train_run(label, over=(), seed=0, data="5h", fraction=1.0, steps=STEPS, probe=PROBE):
    "Returns (metrics of every step, of every probe); reloaded from OUT/<label>.pt when it exists"
    path = OUT / f"{label}.pt"
    if path.exists():
        saved = torch.load(path, weights_only=False)
        print(f"{label}: reloaded from {path}")
        return saved["hist"], saved["probes"]
    run = run_config(over)
    seed_everything(seed)
    m = no_dropout(build_model(run["model"]).to(device).train())
    ds, h = train_set(data, fraction)
    gap = list(itertools.islice(DataLoader(ds, batch_size=BATCH, shuffle=True, generator=torch.Generator().manual_seed(seed + 1000)), GAP_BATCHES))
    n = run["model"].get("num_patches", 1)
    with torch.no_grad(), autocast:
        shape = tuple(m.encode(gap[0][:2].to(device)).shape)
    assert len(shape) == (4 if n > 1 else 3) and (n == 1 or shape[2] == n), (
        f"{label}: states {shape} for num_patches {n}: this code ignores num_patches (sanity step 8 on Colab: both runs were CLS)")
    stream = train_batches(ds, seed)
    print(f"{label}: {' '.join(over) or 'base'} | seed {seed} | {data} x {fraction:g} = {h:.2f} h | {steps} steps | "
          f"lr {run['optim']['lr']:.1e} | sigreg {run['loss']['sigreg']:.3g} | beta {run['loss']['beta']:g}\n"
          f"  states {shape[1:]} per clip | " + " ".join(f"{k} {sum(p.numel() for p in c.parameters()) / 1e6:.2f}M" for k, c in m.named_children()))
    params = [p for p in m.parameters() if p.requires_grad]
    run["iterations"] = steps
    opt, sched = build_optimizer(m, run)  # warmup then cosine decay to 0, as in the real training
    hist, probes = [], {0: probe_all(m, run, gap)}
    for step in range(1, steps + 1):
        before = {n: p.detach().clone() for n, p in m.named_parameters() if p.requires_grad and p.ndim == 2} if step % probe == 0 else None
        t0 = time.perf_counter()
        clip = next(stream).to(device)
        t1 = time.perf_counter()
        lr_now = sched.get_last_lr()[0]
        out = train_step(m, sigreg, clip, opt, sched, params, run, autocast)
        row = {f"train/{key}": v.item() for key, v in out.items()}
        hist.append(row | {"train/lr": lr_now, "system/step_s": time.perf_counter() - t1, "system/data_s": t1 - t0,
                           "system/mem_gb": peak_memory_gb(device) or 0.0})
        if not math.isfinite(row["train/loss"]):
            print(f"{label}: loss not finite at step {step}, run stopped")
            break
        if before is not None:
            probes[step] = probe_all(m, run, gap) | update_ratios(m, before)
            p = probes[step]
            if step % (5 * probe) == 0:
                print(f"{label} step {step} | tf_z_sampled/pers {p['diag/tf_z_sampled / persistence']:.3f} "
                      f"(moving {p['diag/tf_z_sampled / persistence, moving']:.3f}) | kl {p['latent_action/kl']:.3f} | "
                      f"temporal_change {p['representation/temporal_change']:.1e} | {hist[-1]['system/step_s']:.2f} s/step")
    torch.save({"hist": hist, "probes": probes, "cfg": run, "hours": h, "weights": trainable_state(m)}, path)
    del m, opt, sched, params, before, stream
    return hist, probes


def load_net(label):
    "The trained model of a run, eval mode"
    saved = torch.load(OUT / f"{label}.pt", weights_only=False, map_location=device)
    m = build_model(saved["cfg"]["model"]).to(device)
    m.load_state_dict(saved["weights"], strict=False)
    return m.eval()


def boot(num, den, vid, n=1000):
    "sum(num) / sum(den) and its 95 % interval, resampling videos (the clips of a video are not independent)"
    _, inv = np.unique(vid, return_inverse=True)
    sn, sd = np.bincount(inv, num), np.bincount(inv, den)
    idx = np.random.default_rng(0).integers(0, len(sn), (n, len(sn)))
    r = sn[idx].sum(1) / sd[idx].sum(1)
    return num.sum() / den.sum(), *np.quantile(r, [0.025, 0.975])


def stats(p, vid):
    "Final numbers of a probe (a diagnostics dict) with their intervals: (value, low, high)"
    third = tercile(p["raw/cos"])
    out = {}
    for name, b in [("all", third >= 0), ("static", third == 0), ("moving", third == 2)]:
        out[f"tf/pers {name}"] = boot(p["raw/error"][b], p["raw/persistence"][b], vid[b])
    out["retrieval moving"] = boot(p["raw/retrieval"][third == 2], np.ones((third == 2).sum()), vid[third == 2])
    for name, b in [("static", third == 0), ("moving", third == 2)]:
        r = np.median(p["raw/change_ratio"][b])
        out[f"|Δpred|/|Δtrue| {name}"] = (r, r, r)
    return out


def summary(runs, title, noise=None):
    "Final numbers of each run with their 95 % intervals over videos (figure + table). noise: std over seeds (E1)"
    final = {label: stats(pr[max(pr)], val_video) | {key: (pr[max(pr)][key],) * 3 for key in
             ["latent_action/kl", "latent_action/controllability", "representation/temporal_change"]}
             for label, (_, pr) in runs.items()}
    keys = list(next(iter(final.values())))
    fig, axs = plt.subplots(1, 6, figsize=(30, 0.5 * len(runs) + 2), layout="constrained")
    for ax, key in zip(axs, keys[:6]):
        for i, (label, s) in enumerate(final.items()):
            v, lo, hi = s[key]
            ax.errorbar(v, i, xerr=[[v - lo], [hi - v]], fmt="o", color=f"C{i % 10}", capsize=3)
        ax.set_yticks(range(len(final)), list(final) if ax is axs[0] else [])
        ax.invert_yaxis()
        ax.axvline({"retrieval moving": 1 / T}.get(key, 1), color="k", ls=":", lw=1)
        ax.set_title(key + (" (chance 1/T)" if key.startswith("retrieval") else ""), fontsize=10)
    fig.suptitle(f"{title}: end of training, 95 % intervals by bootstrap over the val videos (dotted: persistence, or chance)")
    plt.show()
    print(f"{'run':28}" + "".join(f"{key[:24]:>27}" for key in keys))
    for label, s in final.items():
        cells = []
        for key in keys:
            v, lo, hi = s[key]
            cells.append(f"{v:.3g} [{lo:.3g}, {hi:.3g}]" if hi > lo else f"{v:.3g}")
        print(f"{label[:28]:28}" + "".join(f"{c:>27}" for c in cells))
    if noise:
        print("seed noise (std over the seeds of E1): " + " | ".join(f"{key} {v:.3g}" for key, v in noise.items()))
    return final


def ridge_heldout(x, y, vid):
    '''Ridge regression x -> y fit on the videos of even index (strength by 2-fold cross-validation inside them), tested
    on the others: squared errors per output dimension, and those of the mean of the fit half (no information)'''
    fit = torch.from_numpy(vid % 2 == 0)
    x = (x - x[fit].mean(0)) / (x[fit].std(0) + 1e-6)
    y = y - y[fit].mean(0)

    def errors(a, b, ta, tb, lam):
        n, d = a.shape  # dual form when the features outnumber the samples (states of several tokens)
        w = (torch.linalg.solve(a.T @ a + lam * n * torch.eye(d, dtype=a.dtype), a.T @ ta) if d <= n else
             a.T @ torch.linalg.solve(a @ a.T + lam * n * torch.eye(n, dtype=a.dtype), ta))
        return (b @ w - tb).pow(2).sum(0), (tb - ta.mean(0)).pow(2).sum(0)

    inner = torch.from_numpy(vid[fit.numpy()] % 4 == 0)  # 2 folds of the fit videos
    a, ta = x[fit], y[fit]
    lams = [1e-3, 1e-2, 1e-1, 1.0, 10.0]
    cv = [sum(e.sum() / b.sum() for e, b in [errors(a[inner], a[~inner], ta[inner], ta[~inner], lam),
                                            errors(a[~inner], a[inner], ta[~inner], ta[inner], lam)]) for lam in lams]
    return errors(a, x[~fit], ta, y[~fit], lams[int(np.argmin(cv))])


@torch.no_grad()
def motion_probe(m, grid=4):
    '''Do the states know how much and where the image moves? (MotionJEPA's frozen probes, with the pixels as ground truth:
    SSv2 has no state labels.) A ridge regression from (s_t, Δs, |Δs|), Δs = s_t+1 - s_t, to the mean |pixel change|
    (how much) and, on the transitions whose pixels move more than the median, to its grid x grid map normalized to sum
    1 (where), held out by video (ridge_heldout). NMSE: 1 = no better than the mean, 0 = perfect'''
    feats, amount, where = [], [], []
    for clip in val_batches:
        with autocast:
            e = m.encode(clip.to(device)).float().flatten(2)
        feats.append(torch.cat([e[:, :-1], e[:, 1:] - e[:, :-1], (e[:, 1:] - e[:, :-1]).abs()], -1).flatten(0, 1).cpu())
        d = F.adaptive_avg_pool2d((clip[:, 1:].float() - clip[:, :-1].float()).abs().mean(2) / 255, grid).flatten(0, 1).flatten(1)
        amount.append(d.mean(1, keepdim=True))
        where.append(d / d.sum(1, keepdim=True).clamp_min(1e-8))
    x, amount, where = torch.cat(feats).double(), torch.cat(amount).double(), torch.cat(where).double()
    moving = amount[:, 0] > amount[:, 0].median()
    out = {}
    for name, y, keep in [("how much", amount, torch.ones_like(moving)), ("where", where, moving)]:
        err, base = ridge_heldout(x[keep], y[keep], val_video[keep.numpy()])
        out[f"NMSE {name}"] = (err.sum() / base.sum()).item()
    return out


def refs(key):
    "Reference lines: (value, label)"
    if "/ persistence" in key or key.endswith("std_ratio_z_sampled"):
        return [(1, "persistence")]
    if "/ drift method" in key:
        return [(1, "drift method")]
    if key.startswith("diag/|Δpred| / |Δtrue|"):
        return [(1, "right size")]
    if key == "diag/cos(Δpred, Δtrue)":
        return [(0, "unrelated")]
    if key == "diag/mu_spread / sigma":
        return [(1, "signal = noise")]
    if key.startswith("diag/next-frame retrieval"):
        return [(1 / T, "chance")]
    if key == "representation/temporal_change":
        return [(0.125, "pixels")]
    if key == "train/gnorm":
        return [(cfg["optim"]["grad_clip"], "clip")]
    if key.startswith("diag/update_ratio"):
        return [(-3, "-3")]
    return []


def plot_runs(runs, title):
    "runs: label -> (metrics of every step, of every probe). Every metric, one panel each, bold = the ones that decide"
    series = {}  # metric -> label -> (steps, values)
    for label, (hist, probes) in runs.items():
        for key in hist[0]:
            series.setdefault(key, {})[label] = (np.arange(1, len(hist) + 1), np.array([h[key] for h in hist], dtype=float))
        for key in dict.fromkeys(key for p in probes.values() for key in p if not key.startswith("raw/")):
            xs = [s for s, p in probes.items() if key in p]
            series.setdefault(key, {})[label] = (np.array(xs), np.array([probes[s][key] for s in xs], dtype=float))
    names = [key for key in KEY if key in series] + sorted(key for key in series if key not in KEY)
    cols, rows = 6, math.ceil(len(names) / 6)
    fig, axs = plt.subplots(rows, cols, figsize=(4.4 * cols, 3.2 * rows), layout="constrained")
    for ax, key in zip(axs.flat, names):
        for i, label in enumerate(runs):
            if label in series[key]:
                x, y = series[key][label]
                ax.plot(x, y, color=f"C{i % 10}", lw=1, label=label)
        ys = np.concatenate([y for _, y in series[key].values()])
        ys = ys[np.isfinite(ys)]
        for y, label in refs(key):
            ax.axhline(y, color="k", ls=":", lw=1, label=label)
        if len(ys) and ys.min() > 0 and ys.max() / ys.min() > 30:
            ax.set_yscale("log")
        ax.set_title(key, fontsize=9, fontweight="bold" if key in KEY else "normal")
        ax.tick_params(labelsize=7)
    for ax in axs.flat[len(names):]:
        ax.axis("off")
    axs.flat[0].legend(fontsize=8)
    fig.suptitle(f"{title}: bold = the metrics that decide, x = step")
    plt.show()
    for label, (hist, _) in runs.items():
        clipped = np.mean([h["train/gnorm"] > cfg["optim"]["grad_clip"] for h in hist])
        print(f"{label}: {len(hist)} steps, clipped {clipped:.0%} of the steps, {np.mean([h['system/step_s'] for h in hist]):.2f} s/step")


def plot_transitions(runs, title):
    "Every val transition at the last probe of each run, per decile of similarity of s_t and s_t+1: error / persistence, "
    "direction and size of the predicted change"
    last = {label: pr[max(pr)] for label, (_, pr) in runs.items()}
    fig, axs = plt.subplots(1, 4, figsize=(27, 5.5), layout="constrained")
    for i, (label, p) in enumerate(last.items()):
        c = p["raw/cos"]
        edges = np.quantile(c, np.linspace(0, 1, 11))  # deciles of each run (its own states)
        bins = np.clip(np.digitize(c, edges[1:-1]), 0, 9)
        x = np.arange(10)
        axs[0].plot(x, [p["raw/error"][bins == j].sum() / p["raw/persistence"][bins == j].sum() for j in x], "o-", color=f"C{i % 10}", label=label)
        axs[1].plot(x, [np.median(p["raw/change_cos"][bins == j]) for j in x], "o-", color=f"C{i % 10}", label=label)
        axs[2].plot(x, [np.median(p["raw/change_ratio"][bins == j]) for j in x], "o-", color=f"C{i % 10}", label=label)
        axs[3].plot(x, [p["raw/retrieval"][bins == j].mean() for j in x], "o-", color=f"C{i % 10}", label=label)
    for ax, (ylabel, ref, log) in zip(axs, [("error / persistence", 1, True), ("cos(Δpred, Δtrue), median", 0, False),
                                            ("|Δpred| / |Δtrue|, median", 1, True), ("next-frame retrieval", 1 / T, False)]):
        ax.axhline(ref, color="k", ls=":", lw=1)
        ax.set(xlabel="decile of motion (0 = most moving, 9 = most static)", ylabel=ylabel, yscale="log" if log else "linear")
        ax.legend(fontsize=8)
    fig.suptitle(f"{title}: every val transition at the end, per decile of motion")
    plt.show()


@torch.no_grad()
def transfer(m):
    "z of clip A (the previous clip of the batch) applied to clip B: content leak, motion transfer"
    leak, cos_a, cos_own, moving = [], [], [], []
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device))
            mu = m.idm(emb)[0]
            pred_ab, pred_bb = m.predict(emb[:, :-1], mu.roll(1, 0)), m.predict(emb[:, :-1], mu)
        e, a, p, q = emb.float(), emb.roll(1, 0).float(), pred_ab.float(), pred_bb.float()
        d_a = a[:, 1:] - a[:, :-1]
        leak.append(((p - a[:, 1:]).norm(dim=-1) < (p - e[:, 1:]).norm(dim=-1)).flatten().cpu())
        cos_a.append(F.cosine_similarity(p - e[:, :-1], d_a, dim=-1).flatten().cpu())
        cos_own.append(F.cosine_similarity(q - e[:, :-1], d_a, dim=-1).flatten().cpu())
        moving.append(F.cosine_similarity(a[:, :-1], a[:, 1:], dim=-1).flatten().cpu())
    leak, cos_a, cos_own, moving = map(lambda v: torch.cat(v).float().numpy(), (leak, cos_a, cos_own, moving))
    b = tercile(moving) == 2  # A moves
    return leak.mean(), np.median(cos_a[b]), np.median(cos_own[b])


## E1. Seed noise

**Why.** Residual vs direct flipped between 500 and 1000 steps, and sanity step 8 was in fact CLS against CLS (the
code on Colab ignored num_patches: identical parameter counts), two inits with the same data order, and it gave 0.48
against 0.77 on tf_z_sampled / persistence: the init alone may move our main number by 0.3. **What.** The base run with
three seeds (init and data order; 3 only see large effects, Colas et al.). The β 1e-3 base had five, std 0.072 on
tf/pers, and its spread followed how much z was used (controllability): measured again on β 1e-4. **How to read.** The spread of the three runs, on the curves and on the
intervals, is the smallest difference the next experiments can claim; a run of E4-E7 counts as different from base only
beyond it (tuning playbook). The intervals over videos say how much the val set alone moves a number; the seed spread
adds the training noise. Henderson et al. (2018) split ten seeds of one method into two groups of five and got
two "significantly different" results; Colas et al. (2018) give the number of seeds per effect size (3 seeds see only
large effects).

In [ ]:
seeds = {"base" if s == 0 else f"base, seed {s}": train_run("base" if s == 0 else f"base, seed {s}", seed=s) for s in range(3)}
plot_runs(seeds, "E1, seed noise")
plot_transitions(seeds, "E1, seed noise")
e1 = summary(seeds, "E1, seed noise")
NOISE = {key: float(np.std([s[key][0] for s in e1.values()])) for key in next(iter(e1.values()))}
print("seed noise (std): " + " | ".join(f"{key} {v:.3g}" for key, v in NOISE.items()))

## E2. The invented motion: a constant drift?

**Why.** Sanity 9e: on static clips the prediction moves even with z = mu, its arrows always point the same way. In a
rollout a constant drift adds up at every step. **What.** On every val transition of the base net: Δpred = pred - s_t
with z = mu, z = 0 and the z of another clip, cut into a common part c (the mean of Δpred over all transitions) and the
rest. Compared with the mean true change m (non-zero if the dataset moves on average, e.g. many "moving down").
**How to read.**
- Left: |Δpred| / |Δtrue| per tercile, with and without c. If removing c brings the static tercile near 1, the
  invented motion is the constant drift.
- Second: the share of Δpred along c per tercile. Near 1 on static: they get only the drift.
- Third: c and m per position in the clip. A drift that falls with the position comes from the short context of the
  first predictions; cos(c, m) near 1 means the predictor learned the average motion of the dataset (the MSE answer
  under uncertainty, see E3), a real but unwanted bias: more static data would only rescale it.
- Right: the drift h |c| over a rollout of h steps, against a typical moving step and the distance between two videos.
**Verdict.** Drift ≈ m and the same with any z: a predictor bias, to fix before the rollout (e.g. center the change,
or check the rollout of static clips). Drift that changes with z: the IDM cannot say "nothing moves".
Second figure, the residual diagnostics of forecasting (FPP3 5.4): residuals should have zero mean (else the forecasts
are biased, fix: subtract the mean) and no autocorrelation (else information is left unused). Left, the share of the
error that is a constant bias; middle, the error per tercile on held-out videos before and after subtracting the mean
residual of the other videos (a big drop on static = a constant bias, cheap to fix); right, the autocorrelation of the
residuals along the clip (above the band: the error at t announces the error at t+1, it will pile up in a rollout).

In [ ]:
m = load_net("base")
dp, dt, cos_st, emb_all = {"z = mu": [], "z = 0": [], "z of another clip": []}, [], [], []
with torch.no_grad():
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device))
            mu = m.idm(emb)[0]
            for name, z in [("z = mu", mu), ("z = 0", torch.zeros_like(mu)), ("z of another clip", mu.roll(1, 0))]:
                dp[name].append((m.predict(emb[:, :-1], z).float() - emb[:, :-1].float()).cpu())
        e = emb.float().cpu()
        dt.append(e[:, 1:] - e[:, :-1])
        cos_st.append(F.cosine_similarity(e[:, :-1], e[:, 1:], dim=-1))
        emb_all.append(e)
dp = {name: torch.cat(v) for name, v in dp.items()}  # (clips, T-1, D)
dt, cos_st, emb_all = torch.cat(dt), torch.cat(cos_st), torch.cat(emb_all)
c, mean_true = dp["z = mu"].mean((0, 1)), dt.mean((0, 1))
third = torch.from_numpy(tercile(cos_st.flatten().numpy())).view_as(cos_st)
ntrue = dt.norm(dim=-1)
between = (emb_all[:, 0] - emb_all[:, 0].roll(1, 0)).norm(dim=-1).median()  # two clips of (mostly) different videos

fig, axs = plt.subplots(1, 4, figsize=(28, 5.5), layout="constrained")
w, x = 0.2, np.arange(3)
for i, name in enumerate(dp):
    axs[0].bar(x + (i - 1.5) * w, [(dp[name].norm(dim=-1) / ntrue)[third == j].median().item() for j in x], w, color=f"C{i}", label=name)
axs[0].bar(x + 1.5 * w, [((dp["z = mu"] - c).norm(dim=-1) / ntrue)[third == j].median().item() for j in x], w, color="C0",
           hatch="//", alpha=0.5, label="z = mu, drift c removed")
axs[0].axhline(1, color="k", ls=":", lw=1)
axs[0].set(xticks=x, xticklabels=TERCILES, yscale="log", ylabel="|Δpred| / |Δtrue|, median", title="size of the predicted change per tercile")
axs[0].legend(fontsize=8)
along = (dp["z = mu"] @ (c / c.norm())).pow(2) / dp["z = mu"].pow(2).sum(-1)
axs[1].boxplot([along[third == j].numpy() for j in x], showfliers=False)
axs[1].set(xticklabels=TERCILES, ylabel="share of |Δpred|² along c", title="how much of the predicted change is the common drift (z = mu)")
pos = np.arange(T - 1)
axs[2].plot(pos, dp["z = mu"].mean(0).norm(dim=-1).numpy(), "o-", label="|c_t|: mean Δpred at position t")
axs[2].plot(pos, dt.mean(0).norm(dim=-1).numpy(), "s-", label="|m_t|: mean Δtrue at position t")
axs[2].plot(pos, ntrue.median(0).values.numpy(), "^--", label="median |Δtrue| at position t")
axs[2].set(xlabel="position t of the transition in the clip", ylabel="norm", title=f"drift per position | cos(c, m) = {F.cosine_similarity(c, mean_true, dim=0):.2f}")
axs[2].legend(fontsize=8)
h = np.arange(1, 17)
axs[3].plot(h, h * c.norm().item(), "o-", label="h |c| (coherent drift)")
for y, label, ls in [(ntrue[third == 2].median(), "median moving step", "--"), (ntrue[third == 0].median(), "median static step", ":"),
                     (between, "distance between two videos", "-.")]:
    axs[3].axhline(float(y), color="k", ls=ls, lw=1, label=label)
axs[3].set(xlabel="rollout step h", ylabel="distance", yscale="log", title="drift accumulated over a rollout of a static clip")
axs[3].legend(fontsize=8)
fig.suptitle("E2, the invented motion of the base net, every val transition")
plt.show()
print(f"|c| {c.norm():.3g} | |m| {mean_true.norm():.3g} | cos(c, m) {F.cosine_similarity(c, mean_true, dim=0):.2f} | "
      f"median |Δtrue| static {ntrue[third == 0].median():.3g}, moving {ntrue[third == 2].median():.3g} | between videos {between:.3g}")
for name in dp:
    print(f"{name:18} |Δpred|/|Δtrue| static {(dp[name].norm(dim=-1) / ntrue)[third == 0].median():.2f}, "
          f"moving {(dp[name].norm(dim=-1) / ntrue)[third == 2].median():.2f}")

# residual diagnostics (FPP3 5.4): residuals e = pred - s_t+1 (z = mu) should have zero mean (else biased forecasts) and
# no autocorrelation along the clip (else information left unused, and errors that pile up in a rollout)
res = dp["z = mu"] - dt  # pred - s_t+1 = Δpred - Δtrue
pers_err = dt.pow(2).mean(-1)
even = torch.from_numpy(val_video % 2 == 0).view(third.shape)
bias = res[even].mean(0)  # residual mean of the even videos, the bias adjustment tested on the odd ones
fig, axs = plt.subplots(1, 3, figsize=(22, 4.5), layout="constrained")
share = [(res[third == j].mean(0).pow(2).sum() / res[third == j].pow(2).sum(-1).mean()).item() for j in x]
axs[0].bar(TERCILES, share, color="C3")
axs[0].set(ylabel="|mean residual|² / mean |residual|²", title="bias share of the error (0 = unbiased)")
raw_r = [(res.pow(2).mean(-1)[~even & (third == j)].sum() / pers_err[~even & (third == j)].sum()).item() for j in x]
adj_r = [((res - bias).pow(2).mean(-1)[~even & (third == j)].sum() / pers_err[~even & (third == j)].sum()).item() for j in x]
axs[1].bar(x - 0.2, raw_r, 0.4, label="as predicted")
axs[1].bar(x + 0.2, adj_r, 0.4, label="minus the mean residual (bias adjusted)")
axs[1].axhline(1, color="k", ls=":", lw=1)
axs[1].set(xticks=x, xticklabels=TERCILES, yscale="log", ylabel="error / persistence (odd videos)", title="what a constant bias correction would fix")
axs[1].legend(fontsize=8)
std = (res - res.mean((0, 1))) / res.std((0, 1))
lags = np.arange(1, T - 2)
acf = [(std[:, :-k] * std[:, k:]).mean().item() for k in lags]
band = 1.96 / np.sqrt([res.shape[0] * (T - 1 - k) for k in lags])
axs[2].bar(lags, acf, color="C0")
axs[2].plot(lags, band, "k:", lw=1, label="white noise, 95 % (per clip and lag)")
axs[2].plot(lags, -band, "k:", lw=1)
axs[2].set(xlabel="lag k (transitions)", ylabel="autocorrelation of the residuals", title="residual ACF along the clip (0 = nothing left)")
axs[2].legend(fontsize=8)
fig.suptitle("E2, residual diagnostics (Hyndman & Athanasopoulos, FPP3 5.4), base net, z = mu")
plt.show()
print("bias share per tercile: " + " | ".join(f"{n} {v:.2f}" for n, v in zip(TERCILES, share)))
print("error / persistence, odd videos, raw -> bias adjusted: " + " | ".join(f"{n} {r:.2f} -> {a:.2f}" for n, r, a in zip(TERCILES, raw_r, adj_r)))
print("residual ACF: " + " | ".join(f"lag {k} {v:.3f}" for k, v in zip(lags, acf)))

## E3. The KL weight β, and does z carry motion or content?

**Why.** On moving clips the prediction covers half of the true change. Under MSE this is the optimal answer, not a
bug, when z leaves part of the change uncertain: the conditional mean of an uncertain change is shrunk toward zero
(the blurry mean of Mathieu et al. 2016, the shrinkage forecasts of econometrics). So |Δpred|/|Δtrue| measures how
much z tells, and the question is whether the KL keeps z from telling more. kl is 2.7 nats per transition (about 4
bits). The transfer test is the ξ'-swap test of "Why Latent Actions Fail" (2026): a latent that carries the next
frame's content makes the prediction jump to the other clip's frame. **What.** β = 0, 1e-4 (the base since this sweep), 1e-3 (the base
until then), 1e-2. Then a transfer test on each net:
the z of clip A applied to clip B. **How to read.**
- |Δpred| / |Δtrue| moving (summary) toward 1 when β falls: the KL was the bottleneck.
- Transfer, left: the share of B's predictions that land closer to A's next frame than to B's own: z carries A's
  content (it copies the future instead of describing a motion). Should stay near 0.
- Transfer, right: cos(Δpred of B with z_A, Δtrue of A) against the same with B's own z, on the clips where A moves:
  above it, A's motion carries over to B.
**Verdict.** Pick the largest β that keeps |Δpred|/|Δtrue| moving and transfer, without a content leak. If the best
β is at an end of the range (0 or 1e-2), widen it (cs231n). Sanity check on the way: loss/total_val should rise with β.

In [ ]:
betas = {"β 0": ["loss.beta=0"], "base": [], "β 1e-3": ["loss.beta=0.001"], "β 1e-2": ["loss.beta=0.01"]}
e3_runs = {label: train_run(label, over) for label, over in betas.items()}
plot_runs(e3_runs, "E3, β sweep")
plot_transitions(e3_runs, "E3, β sweep")
summary(e3_runs, "E3, β sweep", globals().get("NOISE"))


res = {label: transfer(load_net(label)) for label in betas}
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
x = np.arange(len(res))
axs[0].bar(x, [r[0] for r in res.values()], color="C3")
axs[0].set(xticks=x, xticklabels=list(res), ylabel="share of transitions", title="content leak: pred(B, z_A) closer to A's next frame than to B's")
axs[1].bar(x - 0.2, [r[1] for r in res.values()], 0.4, label="z of A")
axs[1].bar(x + 0.2, [r[2] for r in res.values()], 0.4, label="B's own z (baseline)")
axs[1].axhline(0, color="k", ls=":", lw=1)
axs[1].set(xticks=x, xticklabels=list(res), ylabel="cos(Δpred of B, Δtrue of A), median", title="motion transfer, A in its moving tercile")
axs[1].legend()
fig.suptitle("E3, transfer test: z of clip A applied to clip B")
plt.show()
for label, (lk, ca, co) in res.items():
    print(f"{label:8} leak {lk:.3f} | cos with z_A {ca:.3f} vs own z {co:.3f}")

## E3, continued: what z and s_t say about s_t+1

**Why.** A transition (s_t -> s_t+1) is mostly the scene, which s_t already gives, plus a change, which only z can give.
So I(s_t; s_t+1) should be large and I(z; s_t+1) small: an action names a change, not a scene. If I(z; s_t+1) gets
close to I(s_t; s_t+1), z carries the next frame itself (the leak of "Why Latent Actions Fail"). The chain rule splits
what z knows: I(z; s_t, s_t+1) = I(z; s_t) + I(z; s_t+1 | s_t), the scene it should not know plus the change it should.
**What.** On every val transition, z sampled as in training (so everything is below the KL, data processing):
- I(s_t; s_t+1), I(z; s_t+1), I(z; s_t) (scene leak), and I(z; s_t+1 | s_t), what z predicts of the part of s_t+1
  that s_t does not (the residual of a ridge s_t -> s_t+1): lower bounds from a ridge probe held out by video,
  Gaussian errors: ½ Σ_d log(variance_d / error_d), in nats per transition.
  Linear probes, so lower bounds: generic estimators (MINE, InfoNCE) are biased in 192 dims and InfoNCE cannot exceed
  log(batch) (Poole et al. 2019, McAllester & Stratos 2020);
- the KL: the upper bound on everything z carries (the rate of Alemi et al. 2018, *Fixing a Broken ELBO*).
**How to read.**
- I(z; s_t+1) far below I(s_t; s_t+1): z is an action, not a frame. Close: leak.
- I(z; s_t) near 0: z does not know the scene (actions that transfer); large: z codes the appearance.
- KL >> I(z; s_t) + I(z; s_t+1 | s_t): z carries bits a linear probe cannot read. The swap of z with another clip
  is not a measure of what the predictor reads (on the β 1e-3 base it gave 51 nats for a KL of 3.2: the penalty of a
  wrong z, not an information); the controllability and the transfer test measure that.
- Right: the KL of each dimension of z. A few dimensions carrying all the rate = the low-dimensional z of sanity 9c.
  Bonheme & Grzes (JMLR 2023): the passive dimensions (KL below 0.01) are uncorrelated in sampled z but correlated in
  mu, the representation the controller and the PCA of 9c use; a high |corr| in mu warns that structure seen in mu may
  come from passive dimensions.

In [ ]:
@torch.no_grad()
def information(m):
    "Nats per val transition, z sampled as in training (see the cell above)"
    m.eval()
    m.idm.train()
    torch.manual_seed(0)
    st, nxt, zs, kl, heads = [], [], [], 0.0, []
    hook = m.idm.head.register_forward_hook(lambda mod, i, o: heads.append(o.float().flatten(0, -2).cpu()))
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device))
            z, k = m.idm(emb)
        e = emb.float().flatten(2)
        st.append(e[:, :-1].flatten(0, 1).cpu())
        nxt.append(e[:, 1:].flatten(0, 1).cpu())
        zs.append(z.float().flatten(0, 1).cpu())
        kl += k.float().item() * len(clip)
    hook.remove()
    m.eval()
    mu, logvar = torch.cat(heads).chunk(2, -1)
    rate = (0.5 * (mu.pow(2) + logvar.clamp(-10, 10).exp() - 1 - logvar.clamp(-10, 10))).mean(0)  # KL per dimension
    passive = rate < 0.01
    corr = lambda v: (torch.corrcoef(v[:, passive].T).abs().sum() - passive.sum()) / max(passive.sum() * (passive.sum() - 1), 1)
    st, nxt, zs = (torch.cat(v).double() for v in (st, nxt, zs))

    def nats(x, y):
        err, base = ridge_heldout(x, y, val_video)
        return 0.5 * (base / err).log().sum().item()

    fit = torch.from_numpy(val_video % 2 == 0)  # what s_t does not explain of s_t+1: residual of a ridge fit on the fit half
    xs, target = (st - st[fit].mean(0)) / (st[fit].std(0) + 1e-6), nxt - nxt[fit].mean(0)
    w = torch.linalg.solve(xs[fit].T @ xs[fit] + 1e-2 * fit.sum() * torch.eye(xs.shape[1], dtype=xs.dtype), xs[fit].T @ target[fit])
    return {"I(s_t; s_t+1)": nats(st, nxt), "I(z; s_t+1)": nats(zs, nxt), "I(z; s_t)": nats(zs, st),
            "I(z; s_t+1 | s_t)": nats(zs, target - xs @ w),
            "KL (upper bound)": kl / len(order),
            "rate": rate.sort(descending=True).values.numpy(), "active dims (KL > 0.01)": int((~passive).sum()),
            "passive |corr|, mu": corr(mu).item() if passive.sum() > 1 else float("nan"),
            "passive |corr|, sampled z": corr(zs.float()).item() if passive.sum() > 1 else float("nan")}


info = {label: information(load_net(label)) for label in betas}
keys = list(next(iter(info.values())))[:5]
fig, axs = plt.subplots(1, 3, figsize=(26, 5), layout="constrained", gridspec_kw={"width_ratios": [1, 3, 2]})
x = np.arange(len(info))
for ax, group in [(axs[0], keys[:1]), (axs[1], keys[1:])]:
    w = 0.8 / len(group)
    for j, key in enumerate(group):
        ax.bar(x + (j - (len(group) - 1) / 2) * w, [v[key] for v in info.values()], w, label=key, color=f"C{keys.index(key)}")
    ax.set(xticks=x, xticklabels=list(info), ylabel="nats per transition")
    ax.axhline(0, color="k", lw=0.5)
    ax.legend(fontsize=8)
axs[0].set_title("what s_t says about s_t+1")
axs[1].set_title("what z says, against what it carries (KL)")
for i, (label, v) in enumerate(info.items()):
    axs[2].plot(np.arange(1, len(v["rate"]) + 1), v["rate"].clip(1e-5), "o-", ms=2, color=f"C{i}", label=label)
axs[2].axhline(0.01, color="k", ls=":", lw=1, label="0.01: passive below")
axs[2].set(xlabel="dimension of z, by decreasing KL", ylabel="KL of the dimension (nats)", yscale="log", title="how the rate is spread over the dimensions")
axs[2].legend(fontsize=8)
fig.suptitle("E3, information per val transition (linear-Gaussian lower bounds held out by video, KL upper bound)")
plt.show()
print(f"{'run':10}" + "".join(f"{key:>24}" for key in keys))
for label, v in info.items():
    print(f"{label:10}" + "".join(f"{v[key]:>24.3f}" for key in keys))
for label, v in info.items():
    passive = (f"mean |corr| between passive dims: in mu {v['passive |corr|, mu']:.3f}, in sampled z "
               f"{v['passive |corr|, sampled z']:.3f}" if len(v["rate"]) - v["active dims (KL > 0.01)"] > 1 else "fewer than 2 passive dims")
    print(f"{label:10} active dims {v['active dims (KL > 0.01)']} | {passive}")

In [ ]:
!pip install -q umap-learn
from huggingface_hub import hf_hub_download
for name in ["5h", "10h", "10h-20cls"]:
    hf_hub_download(f"batgre/h2r-ssv2-{name}", "labels.parquet", repo_type="dataset", local_dir=f"/content/h2r-ssv2-{name}")

## E3, continued: does z mean the same motion in every clip?

**Why.** The transfer test says z_A moves B in A's direction (cos 0.31) and the cycle that an action survives a change
of scene (x1.33 at β 1e-4, x1.03-1.34 in Garrido et al. 2026). Neither says whether the motion makes sense: the same z
should mean "moving down" in every clip, whatever the hand and the object. SSv2 gives that label: the class of each
video, a motion named by a template ("Moving something down", "Pushing something from left to right"...).
**What.** On the moving tercile of the val transitions (the label is the clip's, its static transitions do not show the
motion), the class predicted from mu by a linear probe (ridge on one-hot classes) and by a kNN (k = 10, cosine), fit on
the videos of even index and tested on the others, as the information probes (the val clips are overlapping windows: a
split by clip would find the same transition on both sides). The same probes on s_t (the scene alone) and on
Δs = s_t+1 - s_t (the change of the states, everything z could describe). Then a UMAP of mu, coloured by class (LAPO,
Schmidt & Jiang, ICLR 2024, fig. 1).
**How to read.**
- Accuracy from mu far above chance (the most frequent class of the fit half): z names the motion the same way across
  clips. Near chance: z is a motion only in its own clip, it does not transfer as a meaning.
- mu against Δs: Δs is what the states know of the change; mu close to it = z keeps the class, far below = the IDM
  drops it.
- s_t: how much the scene alone gives the class (the same objects come back in a class). mu above s_t = the class comes
  from the motion, not from a scene leak.
- UMAP: one cloud per class (or per direction, the classes pair up: up / down, left / right) = a structured action
  space; one blob = z continuous with no categories (as in sanity 9c), then read the probes, not the picture.
**Labels.** `labels.parquet` (video, label) at the root of each dataset, from the SSv2 annotation files.

In [ ]:
val_names = [v for p in shards(ROOTS["5h"], "val") for v in pq.read_table(p, columns=["video"]).column("video").to_pylist()]
label_of = dict(zip(*pq.read_table(Path(ROOTS["5h"]) / "labels.parquet", columns=["video", "label"]).to_pydict().values()))
classes = sorted({label_of[v] for v in val_names})
val_class = np.array([classes.index(label_of[val_names[v]]) for v in val_video])  # class of each val transition


@torch.no_grad()
def features(m):
    "Per val transition (eval mode, z = mu): mu, s_t, Δs, and the motion tercile of the model"
    mus, st, ds, cos = [], [], [], []
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device))
            mu = m.idm(emb)[0]
        e = emb.float().flatten(2)
        mus.append(mu.float().flatten(0, 1).cpu())
        st.append(e[:, :-1].flatten(0, 1).cpu())
        ds.append((e[:, 1:] - e[:, :-1]).flatten(0, 1).cpu())
        cos.append(F.cosine_similarity(e[:, :-1], e[:, 1:], dim=-1).flatten().cpu())
    return {"mu": torch.cat(mus).double(), "s_t": torch.cat(st).double(), "Δs": torch.cat(ds).double(),
            "moving": tercile(torch.cat(cos).numpy()) == 2}


def class_probe(x, y, vid, k=10, lam=1e-2):
    "Accuracy of a linear probe (ridge on one-hot classes) and of a kNN (cosine), fit on even videos, tested on odd ones"
    fit = vid % 2 == 0
    x = (x - x[fit].mean(0)) / (x[fit].std(0) + 1e-6)
    a, b, ya, yb = x[fit], x[~fit], torch.from_numpy(y[fit]), torch.from_numpy(y[~fit])
    onehot = F.one_hot(ya, len(classes)).double()
    w = torch.linalg.solve(a.T @ a + lam * len(a) * torch.eye(a.shape[1], dtype=a.dtype), a.T @ (onehot - onehot.mean(0)))
    linear = ((b @ w).argmax(1) == yb).double().mean().item()
    near = (F.normalize(b, dim=1) @ F.normalize(a, dim=1).T).topk(k, dim=1).indices
    knn = (F.one_hot(ya[near], len(classes)).sum(1).argmax(1) == yb).double().mean().item()
    return linear, knn, (yb == ya.bincount().argmax()).double().mean().item()


feats = {label: features(load_net(label)) for label in betas}
coherence = {}
for label, f in feats.items():
    b = f["moving"]
    coherence[label] = {name: class_probe(f[name][b], val_class[b], val_video[b]) for name in ["mu", "s_t", "Δs"]}

fig, axs = plt.subplots(1, 2, figsize=(16, 4.5), layout="constrained")
x = np.arange(len(coherence))
for ax, j, title in [(axs[0], 0, "linear probe"), (axs[1], 1, "kNN, k = 10")]:
    for i, name in enumerate(["mu", "s_t", "Δs"]):
        ax.bar(x + (i - 1) * 0.27, [c[name][j] for c in coherence.values()], 0.27, label=name, color=f"C{i}")
    ax.axhline(np.mean([c["mu"][2] for c in coherence.values()]), color="k", ls=":", lw=1, label="chance (most frequent class)")
    ax.set(xticks=x, xticklabels=list(coherence), ylabel="accuracy on held-out videos", ylim=(0, 1), title=title)
    ax.legend(fontsize=8)
fig.suptitle(f"E3, the SSv2 class ({len(classes)} classes) from z, the scene and the change of the states, moving tercile")
plt.show()
for label, c in coherence.items():
    print(f"{label:8} " + " | ".join(f"{name}: linear {v[0]:.3f}, kNN {v[1]:.3f}" for name, v in c.items()) + f" | chance {c['mu'][2]:.3f}")

import umap

fig, axs = plt.subplots(1, len(feats), figsize=(6 * len(feats), 5.5), layout="constrained")
for ax, (label, f) in zip(axs, feats.items()):
    b = f["moving"]
    xy = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(f["mu"][b].numpy())
    for i, name in enumerate(classes):
        sel = val_class[b] == i
        ax.scatter(xy[sel, 0], xy[sel, 1], s=3, alpha=0.5, color=plt.cm.tab20(i), label=name)
    ax.set(title=f"{label}: UMAP of mu, moving tercile", xticks=[], yticks=[])
axs[-1].legend(fontsize=7, markerscale=4, loc="center left", bbox_to_anchor=(1, 0.5))
fig.suptitle("E3, latent actions coloured by SSv2 class (LAPO fig. 1)")
plt.show()

## E4. The SIGReg weight λ

**Why it is worth a sweep.** λ = 0.09 comes from LeWM (robot sim at their frame rate), never set on our data. It is the
only force against the collapse: too weak, the states get predictable by losing what changes (step 3d: temporal
collapse; 9e: cos(s_t, s_t+1) = 0.9997 while the arm moves), too strong, the states are spread at the expense of the
prediction. Both failures touch the two problems of the rollout (invisible hand, x0.5). MotionJEPA (2026) shows it on
LeWM itself: SIGReg across the batch lets the slow features (score, sky) dominate and suppresses the small fast ones
(Pong ball, Dino cactus), NMSE of a frozen probe 0.66 vs 0.004 with their fix, while the training loss shows nothing.
**What.** λ = 0.03, 0.09 (base),
0.27 (x3 steps), values at batch 128. **How to read.** Compare only normalized numbers: tf/pers (per tercile),
retrieval, |Δpred|/|Δtrue|, temporal_change; loss/total and the raw errors change with λ itself. Plus the motion probe
of the states (how much and where the image moves, NMSE, 1 = blind). **Verdict.** The λ
with the best moving tf/pers and retrieval without temporal collapse; at an end of the range (0.03 or 0.27), widen it
to 0.01 or 0.8 (cs231n: the best value must not sit on the border).

In [ ]:
lambdas = {"λ 0.03": ["loss.sigreg=0.03"], "base": [], "λ 0.27": ["loss.sigreg=0.27"]}
e4_runs = {label: train_run(label, over) for label, over in lambdas.items()}
plot_runs(e4_runs, "E4, SIGReg weight")
plot_transitions(e4_runs, "E4, SIGReg weight")
summary(e4_runs, "E4, SIGReg weight", globals().get("NOISE"))
for label in lambdas:
    print(f"{label:8} motion probe: " + " | ".join(f"{key} {v:.3f}" for key, v in motion_probe(load_net(label)).items()))

## E5. Does the encoder see the hand?

**Why.** In 9e the arm moves and the CLS state barely changes: a ViT-tiny trained from scratch on 5 h may be the real
ceiling, no predictor can predict what the state does not see. **What.** The same run with DINOv2-small frozen
(pretrained on 142 M images), only its projector trained. Then for both nets, on every val transition: the change of
the state against the change of the pixels. **How to read.**
- Summary: moving tf/pers and retrieval of the two encoders.
- Motion probe: NMSE of a frozen ridge probe from the states to how much and where the image moves (1 = blind to it).
- Sensitivity: state change (1 - cos, over its median between two videos, so both encoders share a scale) against the
  mean pixel change; spearman = how well the state follows what moves in the image. A higher curve on the moving
  pixels = the encoder sees more of the motion.
**Verdict.** DINOv2 clearly more sensitive and better: the encoder is the limit (pretraining, or much more video, E6).

In [ ]:
e5_runs = {"base": train_run("base"), "DINOv2 frozen": train_run("DINOv2 frozen", ["model.encoder=configs/models/encoders/dinov2_small.yaml"])}
plot_runs(e5_runs, "E5, encoder")
plot_transitions(e5_runs, "E5, encoder")
summary(e5_runs, "E5, encoder", globals().get("NOISE"))


@torch.no_grad()
def sensitivity(m):
    "Per val transition: mean pixel change, state change over the median change between two videos"
    pix, st, betw = [], [], []
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device)).float()
        f = clip.float() / 255
        pix.append((f[:, 1:] - f[:, :-1]).abs().mean((2, 3, 4)).flatten())
        st.append((1 - F.cosine_similarity(emb[:, :-1], emb[:, 1:], dim=-1)).flatten().cpu())
        betw.append((1 - F.cosine_similarity(emb[:, 0], emb.roll(1, 0)[:, 0], dim=-1)).cpu())
    pix, st = torch.cat(pix).numpy(), torch.cat(st).numpy()
    return pix, st / torch.cat(betw).median().item()


fig, axs = plt.subplots(1, 2, figsize=(15, 5), layout="constrained")
for i, label in enumerate(e5_runs):
    pix, st = sensitivity(load_net(label))
    rho = np.corrcoef(pix.argsort().argsort(), st.argsort().argsort())[0, 1]
    axs[0].scatter(pix, st.clip(1e-6), s=2, alpha=0.15, color=f"C{i}")
    edges = np.quantile(pix, np.linspace(0, 1, 11))
    bins = np.clip(np.digitize(pix, edges[1:-1]), 0, 9)
    centers = [np.median(pix[bins == j]) for j in range(10)]
    meds = [np.median(st[bins == j]) for j in range(10)]
    for ax in axs:
        ax.plot(centers, meds, "o-", color=f"C{i}", label=f"{label} (spearman {rho:.2f})")
for ax in axs:
    ax.set(xlabel="mean |pixel change| between t and t+1", ylabel="state change / change between two videos", xscale="log", yscale="log")
    ax.legend()
axs[0].set_title("every val transition, median per decile of pixel change")
axs[1].set_title("medians only")
fig.suptitle("E5, how much the state follows the image")
plt.show()
for label in e5_runs:
    print(f"{label:14} motion probe: " + " | ".join(f"{key} {v:.3f}" for key, v in motion_probe(load_net(label)).items()))

## E5, continued: where the encoder looks

**Why.** The figure of *Vision Transformers Need Registers* (Darcet et al., ICLR 2024): the attention of the CLS token
in the last layer, over the 16 x 16 patches. Their models put it on a few background patches with a huge norm
("artifacts", global memory) instead of the object; *Vision Transformers Need More Than Registers* (2026) finds the CLS
aggregating background in general. Our state is the CLS token: if it looks at the table and not at the hand, the hand
cannot be in the state. **What.** On the most moving val transitions (one per video): the frame, where the pixels move,
then for each encoder the CLS attention of the last layer (mean over heads, from its queries and keys) and the norm of
each patch token at the output of the last block. **How to read.** Attention on the hand and the moved object = good;
on the background or a few bright isolated patches = the artifacts of the paper (their norm map shows the same spots).
The number: the attention mass on the 10 % most moving patches over 0.1, on every moving transition (1 = no preference
for what moves, above = looks at the motion). Last figure, the first-layer filters (cs231n): the 64 first patch
filters of each encoder. Smooth, diverse edge and color detectors = a trained first layer; noise = not converged, lr
or weight decay off (DINOv2's, pretrained on 142 M images, are the reference).

In [ ]:
import re


@torch.no_grad()
def look(m, clip):
    "CLS attention of the last layer (mean over heads) and token norms at the output of the last block: (B*T, 16, 16) each"
    hf = m.encoder.model
    layers = {n: mod for n, mod in hf.named_modules() if re.search(r"layers?\.\d+$", n)}  # names differ across transformers versions
    last = layers[max(layers, key=lambda n: int(n.rsplit(".", 1)[1]))]
    proj = lambda *ends: next(mod for n, mod in last.named_modules() if n.endswith(ends))
    got = {}
    hooks = [proj("q_proj", "query").register_forward_hook(lambda mod, i, o: got.__setitem__("q", o.float())),
             proj("k_proj", "key").register_forward_hook(lambda mod, i, o: got.__setitem__("k", o.float())),
             last.register_forward_hook(lambda mod, i, o: got.__setitem__("x", (o[0] if isinstance(o, tuple) else o).float()))]
    x = (clip.flatten(0, 1).to(device).float() / 255 - m.encoder.mean) / m.encoder.std
    with autocast:
        hf(pixel_values=x)
    for h in hooks:
        h.remove()
    heads = hf.config.num_attention_heads
    q, k = (got[n].unflatten(-1, (heads, -1)).transpose(1, 2) for n in "qk")  # (N, heads, tokens, d)
    att = (q[:, :, :1] @ k.transpose(-1, -2) / q.shape[-1] ** 0.5).softmax(-1)[:, :, 0].mean(1)  # (N, tokens)
    side = int((x.shape[-1] // hf.config.patch_size))
    n_reg = att.shape[-1] - 1 - side * side  # register tokens, 0 here
    return (att[:, 1 + n_reg:].view(-1, side, side).cpu(), got["x"][:, 1 + n_reg:].norm(dim=-1).view(-1, side, side).cpu())


encoders = {label: load_net(label) for label in e5_runs}
frames_all, moving_maps, looks = [], [], {label: ([], []) for label in encoders}
for clip in val_batches:
    f = clip.float() / 255
    moving_maps.append(F.adaptive_avg_pool2d((f[:, 1:] - f[:, :-1]).abs().mean(2), 16).flatten(0, 1))  # (B*(T-1), 16, 16)
    for label, m in encoders.items():
        att, nrm = look(m, clip)
        looks[label][0].append(att.view(len(clip), T, 16, 16)[:, :-1].flatten(0, 1))  # the frame t of each transition
        looks[label][1].append(nrm.view(len(clip), T, 16, 16)[:, :-1].flatten(0, 1))
motion = torch.cat(moving_maps)
looks = {label: (torch.cat(a), torch.cat(n)) for label, (a, n) in looks.items()}
amount = motion.flatten(1).mean(1)
moving = amount > amount.quantile(2 / 3)
top = motion.flatten(1) >= motion.flatten(1).quantile(0.9, dim=1, keepdim=True)
for label, (att, _) in looks.items():
    share = (att.flatten(1) * top).sum(1) / att.flatten(1).sum(1)
    print(f"{label:14} attention on the 10 % most moving patches / 0.1: {(share[moving] / 0.1).mean():.2f} (1 = no preference)")

# the most moving transitions, one per video
picks, seen = [], set()
for i in amount.argsort(descending=True).tolist():
    if val_video[i] not in seen:
        picks.append(i)
        seen.add(val_video[i])
    if len(picks) == 6:
        break
cols = 2 + 2 * len(encoders)
fig, axs = plt.subplots(len(picks), cols, figsize=(2.6 * cols, 2.6 * len(picks)), layout="constrained")
for r, i in enumerate(picks):
    c, t = divmod(i, T - 1)
    frame = val_batches[c // BATCH][c % BATCH][t].permute(1, 2, 0).numpy()
    panels = [(frame, "frame t", None), (motion[i].numpy(), "|pixel change|", "magma")]
    for label, (att, nrm) in looks.items():
        panels += [(att[i].numpy(), f"{label}: CLS attention", "viridis"), (nrm[i].numpy(), f"{label}: token norm", "viridis")]
    for ax, (img, title, cmap) in zip(axs[r], panels):
        ax.imshow(img, cmap=cmap)
        ax.set_xticks([])
        ax.set_yticks([])
        if r == 0:
            ax.set_title(title, fontsize=9)
fig.suptitle("E5, where each encoder looks, on the most moving val transitions (last layer, 16 x 16 patches)")
plt.show()

fig, axs = plt.subplots(1, len(encoders), figsize=(7 * len(encoders), 7), layout="constrained")
for ax, (label, m) in zip(np.atleast_1d(axs), encoders.items()):
    w = next(mod for mod in m.encoder.model.modules() if isinstance(mod, torch.nn.Conv2d)).weight[:64].detach().float().cpu()
    w = (w - w.amin((1, 2, 3), keepdim=True)) / (w.amax((1, 2, 3), keepdim=True) - w.amin((1, 2, 3), keepdim=True) + 1e-8)
    p = w.shape[-1]
    grid = torch.ones(3, 8 * (p + 1), 8 * (p + 1))
    for i in range(64):
        r, c = divmod(i, 8)
        grid[:, r * (p + 1): r * (p + 1) + p, c * (p + 1): c * (p + 1) + p] = w[i]
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis("off")
    ax.set_title(f"{label}: 64 first patch filters ({p} x {p} px)")
fig.suptitle("E5, first-layer filters (cs231n: smooth and diverse = trained, noise = not converged)")
plt.show()

## E6. Data: how much, and which kind

**Why.** The branch exists to scale the data: is the model limited by it, and should the next hours bring more of the
same actions or new ones? **What.** Five runs of STEPS_DATA steps (longer than the others: at 1000 steps the small sets
are seen a few times only and the curve says little): a quarter and a half of the 5h, the 5h (the videos are nested), 10h (the 5h plus 500 more
videos per class) and 10h with 20 classes (the 5h plus 10 new classes of 500 videos: turning upside down, opening,
closing, rolling, putting into, taking out of, pushing slightly, pushing with something, moving closer, moving away).
Same hours for the last two: quantity against diversity. All measured on the 5h val (classes seen by every run), and at
the end on the val videos of the 10 new classes (seen only by the 20-class run).
Garrido et al. (2026) see the IDM-based prediction improve with data on natural video, but not always downstream;
PhyWorld (Kang et al. 2024) sees video models scale in distribution and not out of it: hence the new-class val.
**How to read.**
- Val against hours (log): still falling at 10 h = data-limited, more video will help; flat = the architecture limits.
- Train-val gap (gap/ curve against val): wide on the small sets = memorization.
- New classes: the 20-class run better there and not worse on the 5h val = diversity generalizes; the 10h run better
  on the 5h val than the 20-class run = for our actions, more of the same pays more.

In [ ]:
STEPS_DATA, PROBE_DATA = 3000, 150
plans = {"data 5h, quarter": ("5h", 0.25), "data 5h, half": ("5h", 0.5), "data 5h": ("5h", 1.0), "data 10h": ("10h", 1.0),
         "data 10h, 20 classes": ("10h-20cls", 1.0)}
e6_runs = {label: train_run(label, data=d, fraction=f, steps=STEPS_DATA, probe=PROBE_DATA) for label, (d, f) in plans.items()}
plot_runs(e6_runs, f"E6, data, {STEPS_DATA} steps")
plot_transitions(e6_runs, "E6, data")
e6 = summary(e6_runs, "E6, data, 5h val", globals().get("NOISE"))

# the val clips of the 10 new classes: val videos of the 20-class set that are not in the 5h set
ds_new = ClipDataset(ROOTS["10h-20cls"], "val", T)
names = lambda root: [v for p in shards(root, "val") for v in pq.read_table(p, columns=["video"]).column("video").to_pylist()]
old = set(names(ROOTS["5h"]))
start, end = videos(ds_new)
new_clips = np.concatenate([np.arange(start[v], end[v]) for v, name in enumerate(names(ROOTS["10h-20cls"])) if name not in old])
new_order = np.random.default_rng(0).permutation(new_clips)[: 4 * BATCH].tolist()
new_batches = list(DataLoader(ds_new, batch_size=BATCH, sampler=new_order))
new_video = np.repeat(np.searchsorted(ds_new.ends, new_order, side="right"), T - 1)
new = {label: stats(diagnostics(load_net(label), new_batches), new_video) for label in plans}

hrs = {label: torch.load(OUT / f"{label}.pt", weights_only=False)["hours"] for label in plans}
fig, axs = plt.subplots(1, 3, figsize=(22, 5), layout="constrained")
for ax, key in zip(axs, ["tf/pers all", "tf/pers moving", "retrieval moving"]):
    for i, label in enumerate(plans):
        for s, marker, dx in [(e6[label], "o", 1.0), (new[label], "s", 1.03)]:
            v, lo, hi = s[key]
            ax.errorbar(hrs[label] * dx, v, yerr=[[v - lo], [hi - v]], fmt=marker, color=f"C{i}", capsize=3,
                        label=f"{label}, {'5h val' if marker == 'o' else 'new classes'}")
    ax.set(xscale="log", xlabel="hours of training video", title=key)
axs[0].legend(fontsize=7)
fig.suptitle("E6: circles = 5h val (classes seen by all), squares = val of the 10 new classes; 95 % intervals over videos")
plt.show()
for label in plans:
    print(f"{label:22} {hrs[label]:.2f} h | new classes: tf/pers {new[label]['tf/pers all'][0]:.3f}, "
          f"moving {new[label]['tf/pers moving'][0]:.3f}, retrieval moving {new[label]['retrieval moving'][0]:.3f}")

## E7. The state: one CLS vector or a few tokens

**Why.** If both CLS states (E5) know how much the image moves but not where (motion probe "where" near 1), a single
vector is the bottleneck. CLS against patches was never measured: in sanity step 8 the code on Colab (data-scale on
GitHub) ignored num_patches, both runs were CLS models (same parameter counts, projector 0.79 M instead of 3.15 M), so
its 0.48 vs 0.77 is init noise, not the projector. train_run now checks the shape of the states and prints the
parameter counts. The projector also differed from the paper it comes from: AD-E2E-JEPA (arxiv 2609.34085) applies two stride-2 convs to the patch
tokens of a *frozen* DINOv3 ViT-L, 16x fewer patches and 4x fewer dims (1024 -> 256: the projector compresses), the
predictor runs on the projected patches (as ours), SIGReg on each patch location and time step across the batch (as
ours), λ 0.09 at batch 128 and 0.025 at 512 (our 1/batch rule), lr 1e-4 at 128 (as ours). What our step 8 did
differently: a ViT-tiny trained from scratch instead of a frozen pretrained dense encoder, a hidden layer of 2048 that
expands 192 -> 2048 -> 192 (3.15 M parameters vs 0.79 M for the CLS MLP), no stop-gradient on the target projection
(they need it with a trainable projector on a frozen encoder; LeWM trains end to end without it).
**What.** All with the direct predictor. At 1000 steps:
- base: CLS (E1);
- patch 4: the 4 x 4 grid average pooled to 2 x 2 tokens, "a few tokens" between the CLS and the grid, hidden 192.
At 3000 steps, the 16-token runs: at 1000 steps their kl was still rising (13 and 10 against 34 for the CLS) and
tf/pers was 0.82, they learn slower than one vector. Measured against the CLS at the same length, "data 5h" of E6
(the base config at 3000 steps, reloaded):
- patch 16, hidden 2048: what step 8 meant to run;
- patch 16: hidden 192, the projector compresses instead of expanding (dims adapted to the paper);
- patch 16, DINOv2 frozen: closest to the paper (frozen pretrained dense encoder, 384 -> 384 -> 192).
The 1000-step runs of patch 16 stay on Drive under their old labels.
**How to read.** tf/pers per tercile, retrieval and |Δpred|/|Δtrue| are per frame (tokens concatenated), comparable
across runs; rankme and the raw errors are not. The motion probe "where" should fall below 1 with tokens: the state
knows where things move. **Verdict.** Tokens that improve "where" and the moving tercile without losing the static
one = the CLS was the bottleneck; patch 4 as good as patch 16 = keep the few tokens (cheaper predictor); the DINOv2
patch run far ahead = the paper's recipe needs its frozen dense encoder.

In [ ]:
STEPS_PATCH, PROBE_PATCH = 3000, 150  # as E6, so "data 5h" is the CLS at the same length
patch16 = ["model.num_patches=16", "model.projector.hidden_dim=192"]
groups = {
    "E7, 1000 steps: CLS or 4 tokens": {
        "base": dict(),
        "patch 4": dict(over=["model.num_patches=4", "model.projector.hidden_dim=192"])},
    f"E7, {STEPS_PATCH} steps: CLS or 16 tokens": {
        "data 5h": dict(steps=STEPS_PATCH, probe=PROBE_PATCH),  # the CLS base at 3000 steps (E6)
        "patch 16, hidden 2048, 3000 steps": dict(over=["model.num_patches=16"], steps=STEPS_PATCH, probe=PROBE_PATCH),
        "patch 16, 3000 steps": dict(over=patch16, steps=STEPS_PATCH, probe=PROBE_PATCH),
        "patch 16, DINOv2 frozen, 3000 steps": dict(over=["model.num_patches=16", "model.projector.hidden_dim=384",
                                                          "model.encoder=configs/models/encoders/dinov2_small.yaml"],
                                                    steps=STEPS_PATCH, probe=PROBE_PATCH)}}
for title, states in groups.items():
    runs = {label: train_run(label, **kw) for label, kw in states.items()}
    plot_runs(runs, title)
    plot_transitions(runs, title)
    summary(runs, title, globals().get("NOISE"))
    for label in states:
        print(f"{label:36} motion probe: " + " | ".join(f"{key} {v:.3f}" for key, v in motion_probe(load_net(label)).items()))

## E8. The rollout loss (K = 2)

**Why.** E1-E5 left a predictor trained only one step ahead from the true past (teacher forcing), with these problems:
- invented motion on static clips: |Δpred|/|Δtrue| 1.8, tf/pers 2.6 on the static third, worse than copying the frame;
  a constant drift c added to every prediction (E2), which a rollout piles up step after step (h |c|);
- shrunk motion on moving clips (|Δpred|/|Δtrue| below 1, the conditional mean of an uncertain change, E3);
- errors left in the residuals (bias share, autocorrelation along the clip, E2);
- a large seed spread, which follows how much z wakes up (E1).
At planning time the predictor reads back its own predictions (W_p = 2 states). Trained on true states only, it never
saw its own errors: the gap between train and test inputs (exposure bias) makes the error grow by a factor Λ per step,
error ~ δ Λ^H (JEPA-WM, Terver et al. 2025, section D). Their k-step loss trains on unrolled predictions: L = tf + L_2,
L_2 the error two steps after a true state, the first prediction detached (last-gradient only), every initial context,
a window of W_t = 3 states in the unrolled step. Their best variant: K = 2. It trades accuracy for robustness: δ can
rise (one step less accurate), Λ should fall.
**What.** The base config with loss.rollout_steps=2 (W_t = loss.rollout_window = 3, W = clip_len - 1 = 7 as the base),
three seeds, against the three seeds of the base (E1, K = 1). Then three cells:
1. training curves, transitions and summary, as the other experiments;
2. every metric of the four nets recomputed the same way (the same code, W_p = 2 for all), problem by problem, against
   the seed spread;
3. the rollouts themselves: error and its growth over the horizon, direction and size of the imagined motion, distance
   to the real states, reading of z deep in the rollout, a long rollout with the "nothing happens" action, the
   trajectories in a plane.
**How to read.** loss/total_val includes L_2 for the K = 2 runs: compare prediction/tf, not the loss. A difference
counts beyond the seed spread of both groups (3 seeds each: only large effects, Colas et al.).
**Verdict.** K = 2 is kept for the first real training if the rollout skill and the growth improve at every horizon
beyond the seed spread, without losing the one-step error by more than the seed spread and without a new problem (z
ignored deep in the rollout, collapse of the temporal change, a content leak). Fixing the static drift is a bonus, the
reason for the rollout loss is the growth.

In [ ]:
k2_labels = ["rollout K2" if s == 0 else f"rollout K2, seed {s}" for s in range(3)]
base_labels = ["base" if s == 0 else f"base, seed {s}" for s in range(3)]
e8_runs = {label: train_run(label, ["loss.rollout_steps=2"], seed=s) for s, label in enumerate(k2_labels)}
e8_runs |= {label: train_run(label, seed=s) for s, label in enumerate(base_labels)}  # E1, reloaded
plot_runs(e8_runs, "E8, rollout loss K = 2 vs teacher forcing")
plot_transitions(e8_runs, "E8, rollout loss K = 2 vs teacher forcing")
summary(e8_runs, "E8, rollout loss K = 2 vs teacher forcing", globals().get("NOISE"))
for label, (hist, _) in e8_runs.items():
    print(f"{label:22} cost: {np.mean([h['system/step_s'] for h in hist[10:]]):.3f} s/step, "
          f"peak memory {max(h['system/mem_gb'] for h in hist):.1f} GB"
          + (f" | L_2 at the end {np.mean([h['train/rollout_2'] for h in hist[-50:]]):.4g}, tf {np.mean([h['train/tf'] for h in hist[-50:]]):.4g}"
             if "train/rollout_2" in hist[0] else ""))

## E8, continued: the old problems and the new ones

**Why.** The probes saved during training differ between the two groups: the base ran before the metrics per third of
pixel motion and the rollout metrics existed. So every metric is recomputed here on the six final nets, the
same code and W_p = 2 for all (the saved base configs have no history_size, they would load W_p = 3).
**What.** For each net: evaluate (prediction, rollout, latent actions, representation, z = mu), the diagnostics of the
tools (z sampled), the drift of E2 (drift/: |c| over a median static step, share of the static Δpred along c, bias share
of the static error, residual autocorrelation at lag 1), the motion probe of E5 (encoder/: does the state know how much
and where the image moves) and the transfer test of E3 (transfer/). Then, per metric: mean of each group, std over its
seeds, and the difference in units of the pooled seed std.
**How to read.** The figure: one panel per metric, a dot per seed, the bar = mean, dotted = the reference (persistence,
right size, chance). Left block, the old problems: change_ratio_static toward 1, tf/pers static toward or below 1,
change_ratio_moving toward 1, |c| and the bias share down, the residual ACF toward 0, a smaller seed spread. Right
block, what K = 2 may break: tf/pers (one step) up, kl, controllability and cycle (z read less: the unrolled step can be
lowered by ignoring z and predicting the mean), the content leak (z copying the next frame shortens the rollout too),
temporal_change and rankme (the encoder also gets the L_2 gradient: an encoder that erases motion makes every rollout
easy), the motion probe, std_ratio (predictions pulled to the mean). The table prints every metric, |d| > 2 flagged.
**Verdict.** A problem is fixed when the K2 dots sit beyond the base dots, all three; a new problem the same way.

In [ ]:
W_P = 2  # the planning window of every net in E8 (model.history_size)


@torch.no_grad()
def drift_stats(m):
    "The numbers of E2 for one net (eval, z = mu)"
    m.eval()
    dp, dt, cos = [], [], []
    for clip in val_batches:
        with autocast:
            emb = m.encode(clip.to(device))
            pred = m.predict(emb[:, :-1], m.idm(emb)[0])
        e = emb.float().flatten(2).cpu()
        dp.append(pred.float().flatten(2).cpu() - e[:, :-1])
        dt.append(e[:, 1:] - e[:, :-1])
        cos.append(F.cosine_similarity(e[:, :-1], e[:, 1:], dim=-1))
    dp, dt, cos = torch.cat(dp), torch.cat(dt), torch.cat(cos)
    static = torch.from_numpy(tercile(cos.flatten().numpy()) == 0).view_as(cos)
    c, res = dp.mean((0, 1)), dp - dt
    std = (res - res.mean((0, 1))) / res.std((0, 1))
    return {"|c| / median static step": (c.norm() / dt.norm(dim=-1)[static].median()).item(),
            "share of Δpred along c, static": ((dp @ (c / c.norm())).pow(2) / dp.pow(2).sum(-1))[static].median().item(),
            "bias share of the error, static": (res[static].mean(0).pow(2).sum() / res[static].pow(2).sum(-1).mean()).item(),
            "residual ACF lag 1": (std[:, :-1] * std[:, 1:]).mean().item()}


def final_metrics(label):
    "Every metric of a trained net, the same code for all (evaluate with K = 1: loss/total_val comparable)"
    m = load_net(label)
    m.history_size = W_P
    out = evaluate(m, sigreg, val_batches, run_config(), device, autocast)
    out |= {key: v for key, v in diagnostics(m, val_batches).items() if not key.startswith("raw/")}
    out |= {f"drift/{key}": v for key, v in drift_stats(m).items()}
    out |= {f"encoder/{key}": v for key, v in motion_probe(m).items()}
    leak, cos_a, cos_own = transfer(m.eval())
    return out | {"transfer/leak": leak, "transfer/cos z_A": cos_a, "transfer/cos own z": cos_own}


e8_groups = {"base (K = 1)": base_labels, "rollout K2": k2_labels}
e8_final = {label: final_metrics(label) for labels in e8_groups.values() for label in labels}
OLD = {"invented motion, static": ("prediction/change_ratio_static", 1), "worse than copying, static": ("prediction/tf_over_persistence_static", 1),
       "shrunk motion, moving": ("prediction/change_ratio_moving", 1), "tf / pers, moving": ("prediction/tf_over_persistence_moving", 1),
       "retrieval, moving": ("prediction/retrieval_moving", 1 / T), "drift |c| / static step": ("drift/|c| / median static step", None),
       "bias share, static": ("drift/bias share of the error, static", 0), "residual ACF lag 1": ("drift/residual ACF lag 1", 0)}
NEW = {"tf / pers, one step": ("prediction/tf_over_persistence", 1), "kl": ("latent_action/kl", None),
       "controllability": ("latent_action/controllability", 0), "cycle": ("latent_action/cycle", 1),
       "scene cut": ("latent_action/scene_cut", 1), "content leak": ("transfer/leak", 0),
       "temporal change": ("representation/temporal_change", None), "rankme": ("representation/rankme", None),
       "motion probe, where": ("encoder/NMSE where", 1), "std ratio": ("prediction/std_ratio", 1)}
panels = list(OLD.items()) + list(NEW.items())
fig, axs = plt.subplots(2, 9, figsize=(36, 8), layout="constrained")
for ax, (title, (key, ref)) in zip(axs.flat, panels):
    for i, (group, labels) in enumerate(e8_groups.items()):
        v = [e8_final[label][key] for label in labels]
        ax.bar(i, np.mean(v), color=f"C{i}", alpha=0.35)
        ax.plot([i] * len(v), v, "o", color=f"C{i}")
    if ref is not None:
        ax.axhline(ref, color="k", ls=":", lw=1)
    ax.set_xticks(range(len(e8_groups)), list(e8_groups), fontsize=8)
    ax.set_title(("old: " if key in dict(OLD.values()) else "new? ") + title, fontsize=9)
for ax in axs.flat[len(panels):]:
    ax.axis("off")
fig.suptitle("E8, every net recomputed the same way (W_p = 2): dots = seeds, bar = mean, dotted = reference")
plt.show()

print(f"{'metric':52}{'base mean (std)':>22}{'K2 mean (std)':>22}{'d':>8}")
for key in e8_final[base_labels[0]]:
    va, vb = (np.array([e8_final[label][key] for label in labels], dtype=float) for labels in e8_groups.values())
    pooled = np.sqrt((va.var(ddof=1) + vb.var(ddof=1)) / 2)
    d = (vb.mean() - va.mean()) / pooled if pooled > 0 else float("nan")
    print(f"{key[:52]:52}{f'{va.mean():.4g} ({va.std(ddof=1):.2g})':>22}{f'{vb.mean():.4g} ({vb.std(ddof=1):.2g})':>22}"
          f"{d:>8.2f}" + ("  <--" if abs(d) > 2 else ""))

## E8, continued: the rollouts, trajectories and coherence

**Why.** The rollout loss is there for the open loop, which the one-step metrics do not see. JEPA-WM measures it as
the error in the embedding space along the unrolled predictor; the planner also needs the imagined trajectory to move
the right way, by the right amount, to stay among real states and to keep obeying z.
**What.** From the first state of every val clip, open loop with W_p = 2 and the IDM's mu of the true transitions,
h = 1..7 (each group: mean of its three seeds, band = min-max over the seeds):
- figure 1, the error: skill(h) = 1 - error / persistence (copy of s_0) on all clips, then on the static and moving
  thirds of the clips' pixel motion; growth(h) = error_h / error_h-1 (Λ); controllability(h) = 1 - error with the true
  z / error with another clip's z: does the predictor still read z deep in the rollout?
- figure 2, the motion: cumulative change ratio |pred_h - s_0| / |s_h - s_0| (above 1 = invented motion piling up,
  below = motion lost), its direction cos(pred_h - s_0, s_h - s_0) on moving clips, the std ratio across clips per h
  (regression to the mean: the rollouts of all clips converging to one state), the retrieval at h (s_h is the frame
  of the clip closest to pred_h, chance 1/T);
- figure 3, the coherence: distance of pred_h to the closest real val state of another video, over the same distance
  for the real s_h (1 = as close to the data as a real state, above = off the states the encoder makes); a 16-step
  rollout with the "nothing happens" action (the IDM's z for the pair (s_0, s_0)) on the static third: the distance to
  s_0, against one moving step and the distance between two videos, and its distance to the data; the straightness
  of the trajectories (cos of consecutive steps, LeWM), predicted against real;
- figure 4, the sensitivity to the planning window W_p = 1, 2, 3, 7 (seed 0 of each group): the paper picks W_p = 2;
- figure 5, three moving clips in the plane of the first two principal components of the val states of each net,
  real trajectory against imagined.
**How to read.** K2 above base on skill at every h and below on growth = the rollout loss works. Controllability
falling to 0 with h = the predictor forgets z in the open loop (a planner could not steer it). Change ratio above 1 and
growing on static clips = the drift of E2 piling up; the no-op rollout leaving s_0 by more than one moving step
within a few steps says the same. Distance to the data above 1 and growing = the rollout leaves the states the
encoder makes: the cost a planner computes there means nothing. Std ratio falling with h = every rollout collapses to
the mean state.
**Verdict.** With figure 1, the verdict of E8; figures 2-3 say why, figure 4 whether W_p = 2 suits our nets too.

In [ ]:
LONG = 16  # steps of the no-op rollout
clip_vid = val_video[:: T - 1]  # video of each val clip


@torch.no_grad()
def rollout_analysis(label, wp=W_P):
    "Open loop from s_0 of every val clip (eval, z = mu of the true transitions), with another clip's z, with the no-op z"
    m = load_net(label)
    m.history_size = wp
    out = {k: [] for k in ["real", "own", "other", "noop", "motion"]}
    for clip in val_batches:
        clip = clip.to(device)
        with autocast:
            emb = m.encode(clip)
            mu = m.idm(emb)[0]
            noop = m.idm(emb[:, [0, 0]])[0]  # the IDM's action between s_0 and itself: nothing happens
            for name, z in [("own", mu), ("other", mu.roll(1, 0)), ("noop", noop.expand(-1, LONG, -1))]:
                out[name].append(m.rollout(emb[:, :1], mu[:, :0], z[:, None])[:, 0].float().flatten(2).cpu())
        out["real"].append(emb.float().flatten(2).cpu())
        out["motion"].append((clip[:, 1:].float() - clip[:, :-1].float()).abs().mean((1, 2, 3, 4)).cpu())
    return {k: torch.cat(v) for k, v in out.items()}


def nearest_real(x, bank, bank_vid, vid):
    "Distance of every state of x (clips, h, D) to the closest real val state of another video"
    out = []
    for i in range(0, len(x), 64):
        d = torch.cdist(x[i : i + 64].flatten(0, 1).to(device), bank.to(device))
        same = torch.from_numpy(vid[i : i + 64, None] == bank_vid[None]).to(device).repeat_interleave(x.shape[1], 0)
        out.append(d.masked_fill(same, float("inf")).min(1).values.view(-1, x.shape[1]).cpu())
    return torch.cat(out)


def straight(traj):
    "Median cos of consecutive steps of trajectories (clips, t, D)"
    v = traj[:, 1:] - traj[:, :-1]
    return F.cosine_similarity(v[:, :-1], v[:, 1:], dim=-1).median().item()


def rollout_curves(r):
    e, own, oth, noop = r["real"], r["own"], r["other"], r["noop"]
    s0, tgt = e[:, :1], e[:, 1:]
    lo, hi = r["motion"].quantile(torch.tensor([1 / 3, 2 / 3]))
    third = {"all": torch.ones(len(e), dtype=torch.bool), "static": r["motion"] <= lo, "moving": r["motion"] >= hi}
    err, pers, err_o = ((p - tgt).pow(2).mean(-1) for p in (own, s0, oth))
    c = {}
    for g, b in third.items():
        c[f"skill {g}"] = 1 - err[b].mean(0) / pers[b].mean(0)
        c[f"change ratio {g}"] = ((own - s0).norm(dim=-1) / (tgt - s0).norm(dim=-1).clamp_min(1e-8))[b].median(0).values
    c["direction moving"] = F.cosine_similarity(own - s0, tgt - s0, dim=-1)[third["moving"]].median(0).values
    c["growth"] = err.mean(0)[1:] / err.mean(0)[:-1]
    c["controllability"] = 1 - err.mean(0) / err_o.mean(0)
    c["std ratio"] = own.var(0).sum(-1).sqrt() / tgt.var(0).sum(-1).sqrt()
    c["retrieval"] = (torch.cdist(own, e).argmin(-1) == torch.arange(1, T)).float().mean(0)
    bank, bank_vid = e.flatten(0, 1), np.repeat(clip_vid, T)
    real_d = nearest_real(tgt, bank, bank_vid, clip_vid).median()
    c["to the data"] = nearest_real(own, bank, bank_vid, clip_vid).median(0).values / real_d
    st = third["static"]
    c["no-op to the data"] = nearest_real(noop[st], bank, bank_vid, clip_vid[st.numpy()]).median(0).values / real_d
    c["no-op |s_h - s_0|"] = (noop[st] - s0[st]).norm(dim=-1).median(0).values
    c["moving step"] = (tgt[:, 1:] - tgt[:, :-1]).norm(dim=-1)[third["moving"]].median()
    c["between videos"] = (e[:, 0] - e[:, 0].roll(1, 0)).norm(dim=-1).median()
    traj = torch.cat([s0, own], 1)
    c["straightness"] = torch.tensor([straight(e[third["moving"]]), straight(traj[third["moving"]])])
    return c


curves = {group: [rollout_curves(rollout_analysis(label)) for label in labels] for group, labels in e8_groups.items()}
h, hl = np.arange(1, T), np.arange(1, LONG + 1)


def band(ax, x, group, key, i, ls="-", label=None):
    v = torch.stack([c[key] for c in curves[group]]).numpy()
    ax.plot(x, v.mean(0), ls, color=f"C{i}", label=label or group, marker="o", ms=3)
    ax.fill_between(x, v.min(0), v.max(0), color=f"C{i}", alpha=0.2)


def finish(ax, title, ref=None, xlabel="horizon h", **kw):
    if ref is not None:
        ax.axhline(ref, color="k", ls=":", lw=1)
    ax.set(title=title, xlabel=xlabel, **kw)
    ax.legend(fontsize=7)


for fig_title, panels in [
    ("E8, figure 1: open-loop error from one real state (W_p = 2)", [
        ("skill(h), all clips (0 = copy s_0)", [("skill all", "-")], 0, {}),
        ("skill(h): static thirds -- , moving thirds -", [("skill static", "--"), ("skill moving", "-")], 0, {}),
        ("growth(h) = error_h / error_h-1", [("growth", "-")], 1, {}),
        ("controllability(h): true z vs another clip's", [("controllability", "-")], 0, {})]),
    ("E8, figure 2: the imagined motion", [
        ("|pred_h - s_0| / |s_h - s_0|: static -- , moving -", [("change ratio static", "--"), ("change ratio moving", "-")], 1, {"yscale": "log"}),
        ("cos(pred_h - s_0, s_h - s_0), moving clips", [("direction moving", "-")], 0, {}),
        ("std across clips, pred / real", [("std ratio", "-")], 1, {}),
        ("retrieval at h (chance 1/T)", [("retrieval", "-")], 1 / T, {})])]:
    fig, axs = plt.subplots(1, 4, figsize=(26, 4.8), layout="constrained")
    for ax, (title, keys, ref, kw) in zip(axs, panels):
        for i, group in enumerate(curves):
            for key, ls in keys:
                x = h[1:] if key == "growth" else h
                band(ax, x, group, key, i, ls, f"{group}, {key}" if len(keys) > 1 else group)
        finish(ax, title, ref, **kw)
    fig.suptitle(fig_title + ": mean of 3 seeds, band = min-max")
    plt.show()

fig, axs = plt.subplots(1, 4, figsize=(26, 4.8), layout="constrained")
for i, group in enumerate(curves):
    band(axs[0], h, group, "to the data", i)
    band(axs[1], hl, group, "no-op |s_h - s_0|", i)
    band(axs[2], hl, group, "no-op to the data", i)
    v = torch.stack([c["straightness"] for c in curves[group]]).numpy()
    axs[3].bar(np.arange(2) + (i - 0.5) * 0.4, v.mean(0), 0.4, color=f"C{i}", label=group, yerr=np.maximum([v.mean(0) - v.min(0), v.max(0) - v.mean(0)], 0))
for key, ls in [("moving step", "--"), ("between videos", "-.")]:
    axs[1].axhline(float(torch.stack([c[key] for c in curves["base (K = 1)"]]).mean()), color="k", ls=ls, lw=1, label=f"{key} (base)")
    axs[1].axhline(float(torch.stack([c[key] for c in curves["rollout K2"]]).mean()), color="gray", ls=ls, lw=1, label=f"{key} (K2)")
finish(axs[0], "distance to the real states of other videos, pred / real", 1)
finish(axs[1], "no-op rollout, static clips: |pred_h - s_0|", xlabel="step h", yscale="log")
finish(axs[2], "no-op rollout: distance to the data, pred / real", 1, xlabel="step h")
axs[3].set_xticks([0, 1], ["real trajectories", "imagined (open loop)"])
finish(axs[3], "straightness, moving clips (cos of consecutive steps)", xlabel="")
fig.suptitle("E8, figure 3: coherence of the rollouts: mean of 3 seeds, band = min-max")
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), layout="constrained")
for ax, (group, labels) in zip(axs, e8_groups.items()):
    for j, wp in enumerate([1, 2, 3, 7]):
        ax.plot(h, rollout_curves(rollout_analysis(labels[0], wp))["skill all"].numpy(), "o-", color=f"C{j + 2}", ms=3, label=f"W_p = {wp}")
    finish(ax, f"{group}, seed 0: skill(h) per planning window", 0)
fig.suptitle("E8, figure 4: sensitivity to W_p (evaluation only, the training is unchanged)")
plt.show()

fig, axs = plt.subplots(2, 3, figsize=(18, 10), layout="constrained")
for row, (group, labels) in zip(axs, e8_groups.items()):
    r = rollout_analysis(labels[0])
    e = r["real"]
    mean = e.flatten(0, 1).mean(0)
    _, _, v = torch.pca_lowrank(e.flatten(0, 1) - mean, q=2, center=False)
    moving = torch.nonzero(r["motion"] >= r["motion"].quantile(0.9)).flatten()[:3]
    for ax, i in zip(row, moving.tolist()):
        real, imag = ((e[i] - mean) @ v).numpy(), ((torch.cat([e[i, :1], r["own"][i]]) - mean) @ v).numpy()
        noop = ((torch.cat([e[i, :1], r["noop"][i]]) - mean) @ v).numpy()
        ax.plot(*real.T, "o-", color="k", label="real")
        ax.plot(*imag.T, "s--", color="C1", label="imagined, true z")
        ax.plot(*noop.T, ".:", color="C2", label=f"imagined, no-op z ({LONG} steps)")
        ax.plot(*real[0], "*", color="r", ms=14)
        ax.set(title=f"{group}, seed 0, val clip {i}", xlabel="PC 1", ylabel="PC 2")
        ax.legend(fontsize=7)
fig.suptitle("E8, figure 5: trajectories of three moving clips (top decile of pixel motion), PCA of each net's val states, star = s_0")
plt.show()

for group, cs in curves.items():
    mean = lambda key: torch.stack([c[key] for c in cs]).mean(0)
    print(f"{group:14} skill h1 {mean('skill all')[0]:.3f} h3 {mean('skill all')[2]:.3f} h7 {mean('skill all')[-1]:.3f} | "
          f"growth mean {mean('growth').mean():.3f} | controllability h7 {mean('controllability')[-1]:.3f} | "
          f"to the data h7 {mean('to the data')[-1]:.2f} | no-op to the data h16 {mean('no-op to the data')[-1]:.2f} | "
          f"straightness real {mean('straightness')[0]:.3f} imagined {mean('straightness')[1]:.3f}")

## Summary of every run

Every run saved in OUT, side by side, with the seed noise of E1. Read a difference only when it is larger than both the
intervals and the seed noise.

In [ ]:
everything = {p.stem: train_run(p.stem) for p in sorted(OUT.glob("*.pt"), key=lambda p: p.stat().st_mtime)}
summary({label: r for label, r in everything.items() if not label.startswith("data ")}, "every 1000-step run", globals().get("NOISE"))
summary({label: r for label, r in everything.items() if label.startswith("data ")}, f"every data run", globals().get("NOISE"));

## References (found from the problems, not the fixes)

- Invented motion and x0.5, the MSE mean: Mathieu, Couprie, LeCun, *Deep multi-scale video prediction beyond mean
  square error*, ICLR 2016 (arxiv 1511.05440); shrinkage of MSE-optimal forecasts: Ashley, *Beyond Optimal
  Forecasting*; copy-last-frame beating learned models on mostly static video: Villegas et al. 2019 (arxiv 1911.01655),
  motion-weighted MSE as evaluation: ACWM-Phys 2026 (arxiv 2605.08567).
- Hand invisible, slow features: MotionJEPA, *Preventing Temporal Feature Collapse by Capturing Visual Changes in Latent
  Space*, 2026 (arxiv 2609.23881), on LeWM's SIGReg; CLS token aggregating background: *Vision Transformers Need
  Registers*, ICLR 2024 (arxiv 2309.16588) and *Vision Transformers Need More Than Registers* (arxiv 2602.22394);
  static bias of SSv2-like data: Kowal et al. CVPR 2022, StillMix (arxiv 2211.12883).
- Patch projector: Zhu, Zhang, LeCun, Choromanska, *AD-E2E-JEPA: A Joint-Embedding Predictive Architecture for End-to-End Autonomous Driving* (arxiv 2609.34085): frozen
  DINOv3, two stride-2 convs, predictor on patches, SIGReg per patch location, stop-gradient on the target projection.
- z copying the future: *Why Latent Actions Fail, and How to Prevent It*, 2026 (arxiv 2605.20223), ξ'-swap test,
  encoder sensitivity bound; Garrido et al., *Learning Latent Action World Models In The Wild*, ICML 2026 (arxiv 2601.05230).
- KL and posterior collapse: free bits / minimum rate, KL annealing, active units (Burda et al. 2016) as diagnostic;
  He et al., *Lagging Inference Networks and Posterior Collapse in VAEs*, ICLR 2019 (arxiv 1901.05534): the latent is
  ignored when the inference network lags early in training, tracked by the mutual information over training;
  Bonheme & Grzes, *Be More Active! Understanding the Differences Between Mean and Sampled Representations of VAEs*,
  JMLR 24 (2023): passive variables are correlated in the mean representation, not in the sampled one.
- Forecast diagnostics: Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed.), 5.2 (naive, mean,
  drift benchmarks), 5.4 (residuals: zero mean, no autocorrelation, Ljung-Box), 5.8 (MASE: our error / persistence).
- Training checks: Karpathy et al., CS231n notes, *Neural Networks Part 3: Learning and Evaluation* (loss at init,
  overfit a tiny set, weights:updates ratio ~1e-3, first-layer filters, lr annealing, best value not at the border of
  a search range, running average of the weights).
- Seed noise: Henderson et al., *Deep RL that Matters*, AAAI 2018 (arxiv 1709.06560); Colas et al., *How Many Random
  Seeds?* (arxiv 1806.08295).
- Rollout, exposure bias: teacher forcing vs own predictions, scheduled sampling, Self Forcing; *Generalization in
  Generation: A closer look at Exposure Bias* (arxiv 1910.00292).